In [114]:
import json
import os
import glob

ruta_base = "../data/raw/2025"

archivos = glob.glob(os.path.join(ruta_base, "*.json"))

len(archivos)

12

In [115]:
todos_los_releases = []

for archivo in archivos:
    with open(archivo, "r", encoding="utf-8") as f:
        paquetes = json.load(f)

    for paquete in paquetes:
        todos_los_releases.extend(paquete["releases"])

len(todos_los_releases)

18326

In [116]:
ocids = [release["ocid"] for release in todos_los_releases]

len(ocids), len(set(ocids))

(18326, 18326)

In [117]:
# Contar cuántos procedimientos tienen información de adjudicación y cuántos no

con_awards = sum(
    1 for release in todos_los_releases
    if "awards" in release and release["awards"]
)

sin_awards = len(todos_los_releases) - con_awards

con_awards, sin_awards

(15718, 2608)

In [118]:
# Contar cuántas entidades contratantes únicas existen en los 18.326 procedimientos

buyers = {
    release["buyer"]["id"]
    for release in todos_los_releases
    if "buyer" in release and release["buyer"].get("id")
}

len(buyers)

1737

In [119]:
# Contar cuántos oferentes únicos aparecen en los procedimientos de 2025

tenderers = set()

for release in todos_los_releases:
    tender = release.get("tender", {})

    for tenderer in tender.get("tenderers", []):
        tenderer_id = tenderer.get("id")
        if tenderer_id:
            tenderers.add(tenderer_id)

len(tenderers)

12125

In [120]:
# Contar cuántos proveedores adjudicados únicos existen en los procedimientos con adjudicación

suppliers = set()

for release in todos_los_releases:
    for award in release.get("awards", []):
        for supplier in award.get("suppliers", []):
            supplier_id = supplier.get("id")
            if supplier_id:
                suppliers.add(supplier_id)

len(suppliers)

5969

In [121]:
# Crear una tabla base con una fila por procedimiento para iniciar el preprocesamiento

import pandas as pd

filas_procedimientos = []

for release in todos_los_releases:
    buyer = release.get("buyer", {})
    tender = release.get("tender", {})

    filas_procedimientos.append({
        "ocid": release.get("ocid"),
        "release_date": release.get("date"),
        "buyer_id": buyer.get("id"),
        "buyer_name": buyer.get("name"),
        "tender_id": tender.get("id"),
        "tender_status": tender.get("status"),
        "number_of_tenderers": tender.get("numberOfTenderers")
    })

procedures_df = pd.DataFrame(filas_procedimientos)

procedures_df.shape

(18326, 7)

In [122]:
# Revisar cuántos valores nulos existen en cada columna de la tabla de procedimientos

procedures_df.isnull().sum()

ocid                     0
release_date             0
buyer_id                 0
buyer_name               0
tender_id                0
tender_status            0
number_of_tenderers    293
dtype: int64

In [123]:














# Mostrar una muestra de procedimientos donde no se informa el número de oferentes

procedures_df[
    procedures_df["number_of_tenderers"].isnull()
].head(10)

,ocid,release_date,buyer_id,buyer_name,tender_id,tender_status,number_of_tenderers
137,ocds-5wno2w-SIE-ASGMCM-2025-001-423644,2025-03-29T06:53:36-05:00,EC-RUC-1768141870001-423644,ACCION SOCIAL DEL GOBIERNO AUTONOMO DESCENTRAL...,SIE-ASGMCM-2025-001-423644,active,NaN
168,ocds-5wno2w-SIE-HAGP-2025-011-17993,2025-04-09T07:58:18-05:00,EC-RUC-0968503870001-17993,Hospital Guayaquil Abel Gilbert Ponton,SIE-HAGP-2025-011-17993,active,NaN
208,ocds-5wno2w-SIE-GADMCEE-2025-002-Z1931401-40764,2025-04-04T07:55:02-05:00,EC-RUC-0968519280001-40764,Municipio del Cantón El Empalme,SIE-GADMCEE-2025-002-Z1931401-40764,active,NaN
257,ocds-5wno2w-SIE-EMAPAL-EP-2025-01-271329,2025-04-09T07:58:09-05:00,EC-RUC-0360027190001-271329,"EMPRESA PUBLICA MUNICIPAL DE AGUA POTABLE, ALC...",SIE-EMAPAL-EP-2025-01-271329,active,NaN
552,ocds-5wno2w-SIE-GADMCN-2025-002-48779,2025-03-21T07:04:47-05:00,EC-RUC-0960006180001-48779,Gobierno Autónomo del Cantón Nobol,SIE-GADMCN-2025-002-48779,active,NaN
559,ocds-5wno2w-SIE-HEJCA-2025-013-87497,2025-04-09T07:58:20-05:00,EC-RUC-0160017400001-87497,HOSPITAL DE ESPECIALIDADES JOSÉ CARRASCO ARTEAGA,SIE-HEJCA-2025-013-87497,active,NaN
722,ocds-5wno2w-SIE-HGGS-2025-003-792270,2025-04-01T07:01:17-05:00,EC-RUC-0968606680001-792270,HOSPITAL GENERAL GUASMO SUR,SIE-HGGS-2025-003-792270,active,NaN
812,ocds-5wno2w-SIE-GADMC_C-2025-00006-2532,2025-04-22T07:21:14-05:00,EC-RUC-0660000520001-2532,GOBIERNO AUTONOMO DESCENTRALIZADO MUNICIPAL DE...,SIE-GADMC_C-2025-00006-2532,active,NaN
1108,ocds-5wno2w-SIE-CDHNSN-2025-00002-1094698,2025-03-22T07:42:15-05:00,EC-RUC-1191706508001-1094698,CONGREGACIÓN DE DOMINICAS HIJAS DE NUESTRA SEÑ...,SIE-CDHNSN-2025-00002-1094698,active,NaN
1125,ocds-5wno2w-SIE-GADMDP-2025-2-38295,2025-03-22T07:42:20-05:00,EC-RUC-0960005370001-38295,Gobierno Palestina,SIE-GADMDP-2025-2-38295,active,NaN


In [124]:
# Comprobar si los procedimientos con numberOfTenderers nulo tienen oferentes registrados en tenderers

casos_nulos = []

for release in todos_los_releases:
    tender = release.get("tender", {})

    if tender.get("numberOfTenderers") is None:
        casos_nulos.append({
            "ocid": release.get("ocid"),
            "tender_status": tender.get("status"),
            "cantidad_tenderers": len(tender.get("tenderers", []))
        })

pd.DataFrame(casos_nulos)["cantidad_tenderers"].value_counts().sort_index()

cantidad_tenderers
0    293
Name: count, dtype: int64

In [125]:
# Revisar los estados de los procedimientos donde no se informa el número de oferentes

pd.DataFrame(casos_nulos)["tender_status"].value_counts(dropna=False)

tender_status
active    293
Name: count, dtype: int64

In [126]:
# Revisar si un mismo identificador de entidad compradora aparece con nombres diferentes

nombres_por_buyer = (
    procedures_df
    .groupby("buyer_id")["buyer_name"]
    .nunique()
)

buyers_con_varios_nombres = nombres_por_buyer[nombres_por_buyer > 1]

len(buyers_con_varios_nombres)

28

In [127]:
# Mostrar ejemplos de buyer_id que aparecen asociados a más de un nombre

ejemplos_variaciones_buyer = (
    procedures_df[
        procedures_df["buyer_id"].isin(buyers_con_varios_nombres.index)
    ]
    .groupby("buyer_id")["buyer_name"]
    .unique()
)

ejemplos_variaciones_buyer.head(10)

buyer_id
EC-RUC-0160001590001-91630     [GOBIERNO AUTONOMO DESCENTRALIZADO MUNICIPAL D...
EC-RUC-0260011290001-36634     [DIRECCION DISTRITAL 02D01-GUARANDA-BOLIVAR-MT...
EC-RUC-0560033630001-346485    [COTOPAXI SOLIDARIO, PATRONATO DE PROTECCION A...
EC-RUC-0660836910001-334029    [EMPRESA PUBLICA EMPRESA MUNICIPAL DE AGUA POT...
EC-RUC-0760008240001-26752     [BENEMERITO CUERPO DE BOMBEROS DE PASAJE, CUER...
EC-RUC-0760026220001-67726     [DIRECCION DISTRITAL 07D02-MACHALA-EL ORO-MTOP...
EC-RUC-0860020000001-89218     [DIRECCION DISTRITAL 08D01-ESMERALDAS-ESMERALD...
EC-RUC-0860034220001-320888    [CUERPO DE BOMBEROS DE ESMERALDAS, BENEMERITO ...
EC-RUC-0968533780001-154032    [CUERPO DE BOMBEROS DE DAULE, BENEMERITO CUERP...
EC-RUC-0968562100001-2572      [FUERZA NAVAL, DIRECCION GENERAL DE TALENTO HU...
Name: buyer_name, dtype: object

In [128]:
# Crear una versión normalizada del nombre de la entidad para facilitar comparaciones,
# conservando siempre el nombre original y utilizando buyer_id como identificador principal

import unicodedata
import re

def normalizar_nombre(texto):
    if pd.isna(texto):
        return texto

    texto = str(texto).strip().upper()
    texto = ''.join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )
    texto = re.sub(r"\s+", " ", texto)

    return texto

procedures_df["buyer_name_normalized"] = (
    procedures_df["buyer_name"].apply(normalizar_nombre)
)

procedures_df[
    ["buyer_name", "buyer_name_normalized"]
].head()

,buyer_name,buyer_name_normalized
0,DIRECCION DISTRITAL 03D02 CAÑAR-EL TAMBO- SUSC...,DIRECCION DISTRITAL 03D02 CANAR-EL TAMBO- SUSC...
1,DIRECCION DISTRITAL DE SALUD # 09D01,DIRECCION DISTRITAL DE SALUD # 09D01
2,DIRECCION DISTRITAL DE SALUD # 09D01,DIRECCION DISTRITAL DE SALUD # 09D01
3,DIRECCION DISTRITAL 04D01-SAN PEDRO DE HUACA-T...,DIRECCION DISTRITAL 04D01-SAN PEDRO DE HUACA-T...
4,GOBIERNO AUTONOMO DESCENTRALIZADO ILUSTRE MUNI...,GOBIERNO AUTONOMO DESCENTRALIZADO ILUSTRE MUNI...


In [129]:
# Comprobar cuántos buyer_id siguen asociados a más de un nombre
# después de eliminar diferencias de tildes, mayúsculas y espacios

nombres_normalizados_por_buyer = (
    procedures_df
    .groupby("buyer_id")["buyer_name_normalized"]
    .nunique()
)

buyers_con_varios_nombres_normalizados = (
    nombres_normalizados_por_buyer[
        nombres_normalizados_por_buyer > 1
    ]
)

len(buyers_con_varios_nombres_normalizados)

27

In [130]:
# Extraer el RUC de 13 dígitos contenido en buyer_id
# para comprobar si un mismo RUC aparece asociado a varios identificadores OCDS

procedures_df["buyer_ruc"] = (
    procedures_df["buyer_id"]
    .str.extract(r"EC-RUC-(\d{13})")
)

ruc_con_varios_ids = (
    procedures_df
    .groupby("buyer_ruc")["buyer_id"]
    .nunique()
)

ruc_con_varios_ids = ruc_con_varios_ids[ruc_con_varios_ids > 1]

len(ruc_con_varios_ids)

0

In [131]:
# Crear una tabla con una fila por procedimiento y oferente participante
# para analizar posteriormente duplicados, IDs, nombres y participación

filas_tenderers = []

for release in todos_los_releases:
    ocid = release.get("ocid")
    tender = release.get("tender", {})

    for tenderer in tender.get("tenderers", []):
        filas_tenderers.append({
            "ocid": ocid,
            "tenderer_id": tenderer.get("id"),
            "tenderer_name": tenderer.get("name")
        })

tender_participation_df = pd.DataFrame(filas_tenderers)

tender_participation_df.shape

(79425, 3)

In [132]:
# Comprobar si existen participaciones duplicadas:
# mismo OCID + mismo tenderer_id repetido más de una vez

duplicados_tenderers = tender_participation_df.duplicated(
    subset=["ocid", "tenderer_id"]
).sum()

duplicados_tenderers

np.int64(0)

In [133]:
# Revisar valores nulos en la tabla de participación de oferentes

tender_participation_df.isnull().sum()

ocid             0
tenderer_id      0
tenderer_name    0
dtype: int64

In [134]:
# Crear una tabla de adjudicaciones con una fila por procedimiento, adjudicación y proveedor
# para analizar montos, fechas, proveedores adjudicados y posibles duplicados

filas_awards = []

for release in todos_los_releases:
    ocid = release.get("ocid")

    for award in release.get("awards", []):
        award_id = award.get("id")
        award_date = award.get("date")

        # Tomar el monto de award.value cuando exista
        award_value = award.get("value", {})
        award_amount = award_value.get("amount")
        award_currency = award_value.get("currency")

        for supplier in award.get("suppliers", []):
            filas_awards.append({
                "ocid": ocid,
                "award_id": award_id,
                "award_date": award_date,
                "supplier_id": supplier.get("id"),
                "supplier_name": supplier.get("name"),
                "award_amount": award_amount,
                "award_currency": award_currency
            })

awards_df = pd.DataFrame(filas_awards)

awards_df.shape

(15718, 7)

In [135]:
# Revisar valores nulos en la tabla de adjudicaciones
# para identificar campos incompletos antes de aplicar transformaciones

awards_df.isnull().sum()

ocid              0
award_id          0
award_date        0
supplier_id       0
supplier_name     0
award_amount      4
award_currency    4
dtype: int64

In [136]:
# Mostrar las adjudicaciones donde falta award.value
# para verificar si existe un campo alternativo como correctedValue

ocids_award_amount_nulo = awards_df.loc[
    awards_df["award_amount"].isnull(),
    "ocid"
].tolist()

for release in todos_los_releases:
    if release.get("ocid") in ocids_award_amount_nulo:
        print("OCID:", release.get("ocid"))
        print(release.get("awards"))
        print("-" * 80)

OCID: ocds-5wno2w-SIE-ECEEP-2025-00004-487227
[{'id': '2532102-SIE-ECEEP-2025-00004', 'date': '2025-10-13T09:22:37-05:00', 'items': [{'id': '4416835-DS-SO', 'unit': {'id': '436', 'name': 'Unidad', 'scheme': 'SERCOP'}, 'quantity': 1, 'description': 'SEDAN', 'classification': {'id': '491130014', 'scheme': 'CPC', 'description': 'SEDAN'}, 'additionalClassifications': [{'id': '49113.00.1', 'uri': 'https://www.compraspublicas.gob.ec/ProcesoContratacion/compras/exe/verComProductos_exe.php?tipo=buscar&idProducto=49113.00.1', 'scheme': 'CPC', 'description': 'AUTOMOVILES'}]}], 'suppliers': [{'id': 'EC-RUC-1791754115001-31585', 'name': 'ASIAUTO S.A'}], 'description': 'Se adjudica por cumplimiento de las especificaciones técnicas', 'enteredValue': {'amount': 14.53499, 'currency': 'USD'}, 'correctedValue': {'amount': 16150, 'currency': 'USD'}}]
--------------------------------------------------------------------------------
OCID: ocds-5wno2w-SIE-DD08D02EAS-2025-016-129463
[{'id': '2533002-SIE-DD08D

In [137]:
# Reconstruir la tabla de adjudicaciones usando award.value como monto principal
# y correctedValue únicamente cuando award.value no esté disponible.
# También se registra la fuente utilizada para mantener trazabilidad.

filas_awards = []

for release in todos_los_releases:
    ocid = release.get("ocid")

    for award in release.get("awards", []):
        award_id = award.get("id")
        award_date = award.get("date")

        if award.get("value"):
            monto = award["value"].get("amount")
            moneda = award["value"].get("currency")
            fuente_monto = "award.value"

        elif award.get("correctedValue"):
            monto = award["correctedValue"].get("amount")
            moneda = award["correctedValue"].get("currency")
            fuente_monto = "award.correctedValue"

        else:
            monto = None
            moneda = None
            fuente_monto = None

        for supplier in award.get("suppliers", []):
            filas_awards.append({
                "ocid": ocid,
                "award_id": award_id,
                "award_date": award_date,
                "supplier_id": supplier.get("id"),
                "supplier_name": supplier.get("name"),
                "award_amount": monto,
                "award_currency": moneda,
                "award_amount_source": fuente_monto
            })

awards_df = pd.DataFrame(filas_awards)

awards_df[
    ["award_amount", "award_currency", "award_amount_source"]
].isnull().sum()

award_amount           0
award_currency         0
award_amount_source    0
dtype: int64

In [138]:
# Contar cuántas adjudicaciones utilizaron award.value
# y cuántas necesitaron correctedValue como fuente alternativa

awards_df["award_amount_source"].value_counts()

award_amount_source
award.value             15714
award.correctedValue        4
Name: count, dtype: int64

In [139]:
# Comprobar si existen montos adjudicados iguales a cero o negativos

montos_cero = (awards_df["award_amount"] == 0).sum()
montos_negativos = (awards_df["award_amount"] < 0).sum()

montos_cero, montos_negativos

(np.int64(0), np.int64(0))

In [140]:
# Revisar las monedas utilizadas en los montos adjudicados
# para comprobar si toda la base está expresada en USD

awards_df["award_currency"].value_counts(dropna=False)

award_currency
USD    15718
Name: count, dtype: int64

In [141]:
# Comprobar si existen adjudicaciones duplicadas:
# mismo procedimiento + misma adjudicación + mismo proveedor

duplicados_awards = awards_df.duplicated(
    subset=["ocid", "award_id", "supplier_id"]
).sum()

duplicados_awards

np.int64(0)

In [142]:
# Revisar si un mismo identificador de proveedor aparece asociado a nombres diferentes

nombres_por_supplier = (
    awards_df
    .groupby("supplier_id")["supplier_name"]
    .nunique()
)

suppliers_con_varios_nombres = nombres_por_supplier[
    nombres_por_supplier > 1
]

len(suppliers_con_varios_nombres)

42

In [143]:
# Mostrar ejemplos de supplier_id que aparecen asociados a más de un nombre

ejemplos_variaciones_supplier = (
    awards_df[
        awards_df["supplier_id"].isin(suppliers_con_varios_nombres.index)
    ]
    .groupby("supplier_id")["supplier_name"]
    .unique()
)

ejemplos_variaciones_supplier.head(10)

supplier_id
EC-RUC-0101324002001-127048    [SARMIENTO JARRIN ANGEL RAMIRO, Sarmiento Jarr...
EC-RUC-0190078825001-67664                  [REPYCOM C. LTDA, REPYCOM CIA. LTDA]
EC-RUC-0190085422001-2884      [RECOR DENTAL Y QUIMEDIC CIA. LTDA, RECOR DENT...
EC-RUC-0190351149001-819042                 [COINT CIA. LTDA., COINT CÍA. LTDA.]
EC-RUC-0200687788001-49352     [Yánez García Yolanda Diocelina, YANEZ GARCIA ...
EC-RUC-0501693477001-475142    [MADRIL VEGA WILLIAMS GIOVANNI, MADRIL VEGA WI...
EC-RUC-0501975254001-519695    [CULQUI DUQUE WILMER GUILLERMO, Culqui Duque W...
EC-RUC-0601105455001-22962     [COELLO VALDIVIESO LUIS ARTURO, Coello Valdivi...
EC-RUC-0601898687001-567835    [Sanchez Mejia Ricardo Javier, SANCHEZ MEJIA R...
EC-RUC-0703388561001-419710    [Ojeda Valarezo Veronica Alicia, OJEDA VALAREZ...
Name: supplier_name, dtype: object

In [144]:
# Crear una versión normalizada del nombre del proveedor
# conservando supplier_id como identificador principal y supplier_name como nombre original

awards_df["supplier_name_normalized"] = (
    awards_df["supplier_name"].apply(normalizar_nombre)
)

# Comprobar cuántos supplier_id siguen asociados a más de un nombre
# después de normalizar mayúsculas, tildes y espacios

nombres_normalizados_por_supplier = (
    awards_df
    .groupby("supplier_id")["supplier_name_normalized"]
    .nunique()
)

suppliers_con_varios_nombres_normalizados = (
    nombres_normalizados_por_supplier[
        nombres_normalizados_por_supplier > 1
    ]
)

len(suppliers_con_varios_nombres_normalizados)

14

In [145]:
# Mostrar los supplier_id que siguen asociados a más de un nombre
# después de la normalización básica, para revisar si son diferencias reales
# o variaciones de puntuación / razón social

ejemplos_supplier_persistentes = (
    awards_df[
        awards_df["supplier_id"].isin(
            suppliers_con_varios_nombres_normalizados.index
        )
    ]
    .groupby("supplier_id")["supplier_name"]
    .unique()
)

ejemplos_supplier_persistentes

supplier_id
EC-RUC-0190078825001-67664                   [REPYCOM C. LTDA, REPYCOM CIA. LTDA]
EC-RUC-0190085422001-2884       [RECOR DENTAL Y QUIMEDIC CIA. LTDA, RECOR DENT...
EC-RUC-0501693477001-475142     [MADRIL VEGA WILLIAMS GIOVANNI, MADRIL VEGA WI...
EC-RUC-0992507896001-711416                     [GARBOCORP S.A.S, GARBOCORP S.A.]
EC-RUC-0992957875001-754300     [SEB TECHNICAL SERVICES CIA.LTDA., SUQUIDAN TE...
EC-RUC-0993366641001-1062338         [CLARYICON S.A.S., ONESCREEN ECUADOR S.A.S.]
EC-RUC-1791314069001-1070       [PURIFLUIDOS, PURIFICACION Y ANALISIS DE FLUID...
EC-RUC-1791737849001-17075      [GRD RADICAL CORPORATION C.L., RADICAL ALTERNA...
EC-RUC-1792335221001-369927     [INGENIERIA Y SERVICIOS AMBIENTALES S.A. ISASA...
EC-RUC-1792693209001-772428                      [CONSTRUCTORAJANOVIC S.A., null]
EC-RUC-1792996635001-954784     [SOLUCIONES TECNOLOGICAS KONECT CIA LTDA, SOLU...
EC-RUC-1793065449001-1030560          [MOONPHARMA CIA LTDA, MOONPHARMA CIA.LTDA.]
EC-R

In [146]:
# Extraer el RUC de 13 dígitos desde supplier_id
# y comprobar si un mismo RUC aparece asociado a varios identificadores OCDS.
# Esto permite detectar posibles fragmentaciones de un mismo proveedor
# que podrían subestimar la concentración. #OBSERVACIÓN ING

awards_df["supplier_ruc"] = (
    awards_df["supplier_id"]
    .str.extract(r"EC-RUC-(\d{13})")
)

ruc_supplier_con_varios_ids = (
    awards_df
    .groupby("supplier_ruc")["supplier_id"]
    .nunique()
)

ruc_supplier_con_varios_ids = (
    ruc_supplier_con_varios_ids[
        ruc_supplier_con_varios_ids > 1
    ]
)

len(ruc_supplier_con_varios_ids)

0

In [147]:
# Identificar proveedores cuyo supplier_id no contiene un RUC ecuatoriano
# de 13 dígitos. Estos casos se revisarán por separado porque podrían
# corresponder a consorcios u otros tipos de identificación.

suppliers_sin_ruc = awards_df[
    awards_df["supplier_ruc"].isna()
]

print("Filas de adjudicación sin RUC estándar:", len(suppliers_sin_ruc))
print("Proveedores únicos sin RUC estándar:", suppliers_sin_ruc["supplier_id"].nunique())

Filas de adjudicación sin RUC estándar: 205
Proveedores únicos sin RUC estándar: 55


In [148]:
# Mostrar los proveedores únicos que no tienen un RUC estándar de 13 dígitos
# para identificar si corresponden a consorcios u otros tipos de actor

proveedores_sin_ruc_unicos = (
    suppliers_sin_ruc[
        ["supplier_id", "supplier_name"]
    ]
    .drop_duplicates()
    .sort_values("supplier_name")
)

proveedores_sin_ruc_unicos.head(30)

,supplier_id,supplier_name
12241,ID-1900219104001-18428,ALVARADO LOZANO XAVIER ALONSO
12548,ID-1792458455001-506201,ANTONOIL SERVICE COMPANY S.A.
380,ID-1758225484001-946355,ARISTIZABAL HENAO JHON ALEXANDER
12982,ID-1103983118001-847870,ARTEAGA MORALES ANGEL ESTUARDO
1253,ID-0491522062001-850355,ASOCIACION DE SERVICIOS LIMPIEZA DE PERSONAS C...
12404,ID-1719405605001-1064018,BIEDMA RAUL ANTONIO
12403,ID-1792306787001-452158,BIOADVANCED CIA. LTDA.
4394,ID-1793197606001-1163344,BLAU FARMACEUTICA ECUADOR S.A.
5845,ID-1758307621001-1219981,CARDOZO MONTENEGRO ERIKA LIZETH
1048,ID-0991298657001-57782,"CECUAMAQ, CORPORACION ECUATORIANA INDUSTRIAL D..."


In [149]:
# Extraer el RUC de 13 dígitos tanto de identificadores EC-RUC- como ID-.
# Se conserva supplier_id original y se crea supplier_ruc como clave auxiliar
# para detectar si un mismo proveedor aparece con diferentes prefijos o IDs.

awards_df["supplier_ruc"] = (
    awards_df["supplier_id"]
    .str.extract(r"(?:EC-RUC-|ID-)(\d{13})")
)

print("Filas sin RUC identificable:", awards_df["supplier_ruc"].isna().sum())
print(
    "Proveedores únicos sin RUC identificable:",
    awards_df.loc[
        awards_df["supplier_ruc"].isna(),
        "supplier_id"
    ].nunique()
)

Filas sin RUC identificable: 0
Proveedores únicos sin RUC identificable: 0


In [150]:
# Comprobar si un mismo RUC aparece asociado a más de un supplier_id.
# Esto permite detectar si un mismo proveedor está fragmentado en varios identificadores,
# lo que podría subestimar la concentración en los análisis posteriores.

ruc_con_varios_supplier_ids = (
    awards_df
    .groupby("supplier_ruc")["supplier_id"]
    .nunique()
)

ruc_con_varios_supplier_ids = (
    ruc_con_varios_supplier_ids[
        ruc_con_varios_supplier_ids > 1
    ]
)

len(ruc_con_varios_supplier_ids)

0

In [151]:
# Identificar proveedores adjudicados cuyo nombre contiene referencias a consorcios.
# Estos casos se revisan por separado porque un consorcio puede representar
# una asociación temporal de varios participantes y no debe fusionarse
# automáticamente con una empresa individual sin evidencia adicional.

consorcios_df = awards_df[
    awards_df["supplier_name_normalized"]
    .fillna("")
    .str.contains("CONSORC", case=False)
]

print("Filas de adjudicación asociadas a consorcios:", len(consorcios_df))
print("Consorcios únicos por RUC:", consorcios_df["supplier_ruc"].nunique())

Filas de adjudicación asociadas a consorcios: 24
Consorcios únicos por RUC: 8


In [152]:
# Mostrar los consorcios únicos identificados en las adjudicaciones
# para revisar sus nombres y RUC antes de definir su tratamiento metodológico

consorcios_unicos = (
    consorcios_df[
        ["supplier_ruc", "supplier_id", "supplier_name"]
    ]
    .drop_duplicates()
    .sort_values("supplier_name")
)

consorcios_unicos

,supplier_ruc,supplier_id,supplier_name
3317,1792979420001,EC-RUC-1792979420001-1019041,CONSORCIO CORPORACION SEHIDEC - SERVICIOS HIDR...
697,0195148023001,EC-RUC-0195148023001-1225935,CONSORCIO CUATRO RIOS
584,1391901739001,EC-RUC-1391901739001-1081405,CONSORCIO ECUATORIANO DE REDES Y SATELITES CON...
3017,1791251237001,EC-RUC-1791251237001-79499,CONSORCIO ECUATORIANO DE TELECOMUNICACIONES S....
8131,0891743793001,EC-RUC-0891743793001-606011,CONSORCIO ESMERALDEÑO DE METALES TOSCANO COESM...
4245,0691786410001,EC-RUC-0691786410001-1231484,CONSORCIO TRANSPORTE TOTAL
4779,1391938435001,EC-RUC-1391938435001-1273317,CONSORCIO UNIDAD RENTA CAR C.V.
16,0691786383001,EC-RUC-0691786383001-1232133,CONSORCIO VICTORIA UNO


In [153]:
# Revisar si los registros de SERCOP contienen información adicional
# sobre los consorcios dentro de la sección parties.
# El objetivo es determinar si es posible identificar formalmente
# a sus integrantes antes de decidir cualquier unificación.

rucs_consorcios = set(consorcios_df["supplier_ruc"].dropna())

parties_consorcios = []

for release in todos_los_releases:
    for party in release.get("parties", []):
        party_id = party.get("id", "")

        ruc_encontrado = None
        match = re.search(r"(?:EC-RUC-|ID-)(\d{13})", str(party_id))
        if match:
            ruc_encontrado = match.group(1)

        if ruc_encontrado in rucs_consorcios:
            parties_consorcios.append(party)

len(parties_consorcios)

67

In [154]:
# Revisar qué campos están disponibles en los registros "parties"
# asociados a los consorcios.
# Buscamos si existe alguna variable que identifique formalmente
# miembros, integrantes o relaciones del consorcio con otras empresas.

campos_consorcios = sorted(
    set().union(*(party.keys() for party in parties_consorcios))
)

campos_consorcios

['address', 'contactPoint', 'id', 'identifier', 'name', 'roles']

In [155]:
# Convertir la fecha de adjudicación a formato datetime
# y revisar el rango temporal real de las adjudicaciones.
# Esto permitirá documentar si existen adjudicaciones anteriores o posteriores a 2025.

awards_df["award_date_dt"] = pd.to_datetime(
    awards_df["award_date"],
    utc=True
)

awards_df["award_date_dt"].min(), awards_df["award_date_dt"].max()

(Timestamp('2025-01-30 14:52:37+0000', tz='UTC'),
 Timestamp('2026-07-06 15:18:48+0000', tz='UTC'))

In [156]:
# Contar las adjudicaciones según el año de su fecha.
# Esto permite identificar cuántos resultados corresponden a 2025
# y cuántos fueron adjudicados posteriormente, para definir y justificar
# correctamente el corte temporal del análisis.

awards_df["award_year"] = awards_df["award_date_dt"].dt.year

awards_df["award_year"].value_counts().sort_index()

award_year
2025    13859
2026     1859
Name: count, dtype: int64

In [157]:
# Revisar el año de publicación de los releases integrados.
# Esto permite diferenciar la fecha del registro publicado por SERCOP
# de la fecha específica de adjudicación del procedimiento.

procedures_df["release_date_dt"] = pd.to_datetime(
    procedures_df["release_date"],
    utc=True
)

procedures_df["release_year"] = procedures_df["release_date_dt"].dt.year

procedures_df["release_year"].value_counts().sort_index()

release_year
2025    11735
2026     6591
Name: count, dtype: int64

In [158]:
# Extraer la fecha de inicio del período de licitación de cada procedimiento.
# Esta variable permitirá verificar el año real de inicio de los procesos
# y diferenciarlo de la fecha de publicación del release y de la adjudicación.

fechas_inicio_tender = []

for release in todos_los_releases:
    tender = release.get("tender", {})
    tender_period = tender.get("tenderPeriod", {})

    fechas_inicio_tender.append({
        "ocid": release.get("ocid"),
        "tender_start_date": tender_period.get("startDate")
    })

tender_dates_df = pd.DataFrame(fechas_inicio_tender)

tender_dates_df["tender_start_date_dt"] = pd.to_datetime(
    tender_dates_df["tender_start_date"],
    utc=True,
    errors="coerce"
)

print("Fechas de inicio nulas:", tender_dates_df["tender_start_date_dt"].isna().sum())

tender_dates_df["tender_start_date_dt"].dt.year.value_counts().sort_index()

Fechas de inicio nulas: 0


tender_start_date_dt
2025    18299
2026       27
Name: count, dtype: int64

In [159]:
# Revisar los 27 procedimientos cuyo tenderPeriod.startDate corresponde a 2026.
# El objetivo es determinar por qué aparecen dentro del conjunto de datos 2025
# antes de decidir si deben conservarse o excluirse del análisis.

ocids_inicio_2026 = tender_dates_df.loc[
    tender_dates_df["tender_start_date_dt"].dt.year == 2026,
    "ocid"
]

procedimientos_inicio_2026 = procedures_df[
    procedures_df["ocid"].isin(ocids_inicio_2026)
][
    ["ocid", "release_date", "buyer_name", "tender_id", "tender_status"]
]

procedimientos_inicio_2026

,ocid,release_date,buyer_name,tender_id,tender_status
6342,ocds-5wno2w-SIE-CBCG-2025-002-199773,2026-05-27T18:32:15-05:00,CUERPO DE BOMBEROS DEL CANTON GUANO,SIE-CBCG-2025-002-199773,active
6350,ocds-5wno2w-SIE-EPP-2025-336-253178,2026-02-06T07:18:31-05:00,Empresa Pública de hidrocarburos del Ecuador E...,SIE-EPP-2025-336-253178,active
6352,ocds-5wno2w-SIE-GADCM-2025-004-21126,2026-07-04T11:41:36-05:00,GOBIERNO MUNICIPAL DE MOCACHE,SIE-GADCM-2025-004-21126,active
6359,ocds-5wno2w-SIE-GADMA-2025-095-2161,2026-02-13T07:23:59-05:00,GOBIERNO AUTONOMO DESCENTRALIZADO MUNICIPALIDA...,SIE-GADMA-2025-095-2161,active
6360,ocds-5wno2w-SIE-GADMA-2025-094-2161,2026-05-27T18:35:35-05:00,GOBIERNO AUTONOMO DESCENTRALIZADO MUNICIPALIDA...,SIE-GADMA-2025-094-2161,active
6362,ocds-5wno2w-SIE-GADMR-2025-078-33554,2026-02-07T07:25:32-05:00,GOBIERNO AUTONOMO DESCENTRALIZADO DEL CANTON R...,SIE-GADMR-2025-078-33554,active
6363,ocds-5wno2w-SIE-GADMR-2025-080-33554,2026-05-27T18:36:53-05:00,GOBIERNO AUTONOMO DESCENTRALIZADO DEL CANTON R...,SIE-GADMR-2025-080-33554,active
6364,ocds-5wno2w-SIE-GADMR-2025-077-33554,2026-07-01T20:42:26-05:00,GOBIERNO AUTONOMO DESCENTRALIZADO DEL CANTON R...,SIE-GADMR-2025-077-33554,active
6366,ocds-5wno2w-SIE-GADTIXAN-2025-006-116619,2026-05-27T18:37:46-05:00,GOBIERNO AUTONOMO DESCENTRALIZADO PARROQUIAL R...,SIE-GADTIXAN-2025-006-116619,complete
6377,ocds-5wno2w-SIE-GADMR-2025-079-33554,2026-03-04T08:05:55-05:00,GOBIERNO AUTONOMO DESCENTRALIZADO DEL CANTON R...,SIE-GADMR-2025-079-33554,active


In [160]:
# Mostrar las fechas de inicio de los 27 procedimientos cuyo tenderPeriod.startDate
# aparece en 2026, junto con su OCID y tender_id.
# Esto permite revisar si se trata de procesos identificados como 2025
# pero actualizados o iniciados formalmente en 2026.

revision_27 = tender_dates_df[
    tender_dates_df["ocid"].isin(ocids_inicio_2026)
].merge(
    procedures_df[
        ["ocid", "tender_id", "release_date", "tender_status"]
    ],
    on="ocid",
    how="left"
)

revision_27[
    ["ocid", "tender_id", "tender_start_date", "release_date", "tender_status"]
]

,ocid,tender_id,tender_start_date,release_date,tender_status
0,ocds-5wno2w-SIE-CBCG-2025-002-199773,SIE-CBCG-2025-002-199773,2025-12-31T19:00:00-05:00,2026-05-27T18:32:15-05:00,active
1,ocds-5wno2w-SIE-EPP-2025-336-253178,SIE-EPP-2025-336-253178,2025-12-31T19:00:00-05:00,2026-02-06T07:18:31-05:00,active
2,ocds-5wno2w-SIE-GADCM-2025-004-21126,SIE-GADCM-2025-004-21126,2025-12-31T20:00:00-05:00,2026-07-04T11:41:36-05:00,active
3,ocds-5wno2w-SIE-GADMA-2025-095-2161,SIE-GADMA-2025-095-2161,2025-12-31T20:00:00-05:00,2026-02-13T07:23:59-05:00,active
4,ocds-5wno2w-SIE-GADMA-2025-094-2161,SIE-GADMA-2025-094-2161,2025-12-31T19:00:00-05:00,2026-05-27T18:35:35-05:00,active
5,ocds-5wno2w-SIE-GADMR-2025-078-33554,SIE-GADMR-2025-078-33554,2025-12-31T20:00:00-05:00,2026-02-07T07:25:32-05:00,active
6,ocds-5wno2w-SIE-GADMR-2025-080-33554,SIE-GADMR-2025-080-33554,2025-12-31T20:00:00-05:00,2026-05-27T18:36:53-05:00,active
7,ocds-5wno2w-SIE-GADMR-2025-077-33554,SIE-GADMR-2025-077-33554,2025-12-31T20:00:00-05:00,2026-07-01T20:42:26-05:00,active
8,ocds-5wno2w-SIE-GADTIXAN-2025-006-116619,SIE-GADTIXAN-2025-006-116619,2025-12-31T20:00:00-05:00,2026-05-27T18:37:46-05:00,complete
9,ocds-5wno2w-SIE-GADMR-2025-079-33554,SIE-GADMR-2025-079-33554,2025-12-31T20:00:00-05:00,2026-03-04T08:05:55-05:00,active


In [161]:
# Extraer el año del tenderPeriod.startDate respetando la fecha local
# registrada por SERCOP, sin convertir previamente la hora a UTC.
# Esto evita que procedimientos del 31 de diciembre de 2025
# sean clasificados artificialmente como 2026 por el cambio de zona horaria.

tender_dates_df["tender_start_year_local"] = (
    tender_dates_df["tender_start_date"]
    .str[:4]
    .astype(int)
)

tender_dates_df["tender_start_year_local"].value_counts().sort_index()

tender_start_year_local
2025    18326
Name: count, dtype: int64

In [162]:
# Extraer el año de adjudicación respetando la fecha local registrada por SERCOP.
# Se evita convertir previamente a UTC para no desplazar fechas cercanas
# al cambio de año y clasificar incorrectamente una adjudicación.

awards_df["award_year_local"] = (
    awards_df["award_date"]
    .str[:4]
    .astype(int)
)

awards_df["award_year_local"].value_counts().sort_index()

award_year_local
2025    13860
2026     1858
Name: count, dtype: int64

In [163]:
# Revisar cómo se encuentra registrado el valor referencial del procedimiento.
# Se verifica si existe tender.value y cuántos lotes contiene cada procedimiento,
# antes de definir qué campo monetario utilizar como valor referencial.

revision_valor_tender = []

for release in todos_los_releases:
    tender = release.get("tender", {})
    lots = tender.get("lots", [])

    revision_valor_tender.append({
        "ocid": release.get("ocid"),
        "tiene_tender_value": tender.get("value") is not None,
        "numero_lotes": len(lots)
    })

revision_valor_tender_df = pd.DataFrame(revision_valor_tender)

print(
    "Procedimientos con tender.value:",
    revision_valor_tender_df["tiene_tender_value"].sum()
)

print(
    "Distribución del número de lotes:"
)

revision_valor_tender_df["numero_lotes"].value_counts().sort_index()

Procedimientos con tender.value: 0
Distribución del número de lotes:


numero_lotes
0        1
1    18325
Name: count, dtype: int64

In [164]:
# Identificar el único procedimiento que no contiene lotes.
# Se revisará por separado antes de decidir cómo tratar su valor referencial.

procedimiento_sin_lote = revision_valor_tender_df[
    revision_valor_tender_df["numero_lotes"] == 0
]

procedimiento_sin_lote

,ocid,tiene_tender_value,numero_lotes
2569,ocds-5wno2w-SIE-CNELM-017A-2011-124705,False,0


In [165]:
# Revisar el único procedimiento que no contiene tender.lots.
# El objetivo es identificar si el valor referencial aparece en otro campo
# antes de decidir si se mantiene como nulo o se utiliza una fuente alternativa.

ocid_sin_lote = "ocds-5wno2w-SIE-CNELM-017A-2011-124705"

for release in todos_los_releases:
    if release.get("ocid") == ocid_sin_lote:
        print("OCID:", release.get("ocid"))
        print("Fecha release:", release.get("date"))
        print("Tender:")
        print(release.get("tender"))
        print("\nPlanning:")
        print(release.get("planning"))
        print("\nAwards:")
        print(release.get("awards"))
        break

OCID: ocds-5wno2w-SIE-CNELM-017A-2011-124705
Fecha release: 2025-05-24T10:29:59-05:00
Tender:
{'id': 'SIE-CNELM-017A-2011-124705', 'title': 'SIE-CNELM-017A-2011-124705', 'status': 'active', 'awardPeriod': {'endDate': '2025-05-27T12:30:00-05:00', 'startDate': '2025-05-26T12:00:00-05:00', 'maxExtentDate': '2025-05-27T12:30:00-05:00', 'durationInDays': 1}, 'description': 'PRUEBA', 'tenderPeriod': {'startDate': '2025-05-05T15:00:00-05:00'}, 'awardCriteria': 'ratedCriteria', 'enquiryPeriod': {'endDate': '2025-05-07T15:00:00-05:00', 'startDate': '2025-05-05T15:00:00-05:00', 'maxExtentDate': '2025-05-07T15:00:00-05:00', 'durationInDays': 2}, 'procuringEntity': {'id': 'EC-RUC-09925984680012-124705', 'name': 'CNELMANABI'}, 'procurementMethod': 'open', 'mainProcurementCategory': 'goods', 'procurementMethodDetails': 'Subasta Inversa Electrónica'}

Planning:
{'budget': {'id': '5.2.1.3.10.001'}, 'rationale': 'PRUEBA'}

Awards:
None


In [166]:
# Validar estrictamente los identificadores de las entidades compradoras.
# Un RUC válido para este control debe contener exactamente 13 dígitos
# entre el prefijo EC-RUC- y el siguiente guion.
# Esto evita aceptar accidentalmente identificadores con 14 o más dígitos.

procedures_df["buyer_ruc_valido"] = (
    procedures_df["buyer_id"]
    .str.extract(r"^EC-RUC-(\d{13})-")[0]
)

print(
    "Procedimientos con buyer_id sin RUC válido de 13 dígitos:",
    procedures_df["buyer_ruc_valido"].isna().sum()
)

procedures_df[
    procedures_df["buyer_ruc_valido"].isna()
][
    ["ocid", "buyer_id", "buyer_name", "tender_id", "tender_status"]
]

Procedimientos con buyer_id sin RUC válido de 13 dígitos: 1


,ocid,buyer_id,buyer_name,tender_id,tender_status
2569,ocds-5wno2w-SIE-CNELM-017A-2011-124705,EC-RUC-09925984680012-124705,CNELMANABI,SIE-CNELM-017A-2011-124705,active


In [167]:
# Validar estrictamente los identificadores de los proveedores adjudicados.
# Se acepta un RUC de exactamente 13 dígitos tanto para prefijos EC-RUC- como ID-.
# Esto permite detectar identificadores mal formados antes de calcular concentración.

awards_df["supplier_ruc_validado"] = (
    awards_df["supplier_id"]
    .str.extract(r"^(?:EC-RUC-|ID-)(\d{13})-")[0]
)

print(
    "Adjudicaciones con supplier_id sin RUC válido de 13 dígitos:",
    awards_df["supplier_ruc_validado"].isna().sum()
)

awards_df[
    awards_df["supplier_ruc_validado"].isna()
][
    ["ocid", "supplier_id", "supplier_name"]
]

Adjudicaciones con supplier_id sin RUC válido de 13 dígitos: 23


,ocid,supplier_id,supplier_name
1706,ocds-5wno2w-SIE-CEE-2025-019-2504,EC-RUC-17924702930013-513704,UNION CEMENTERA NACIONAL UCEM S.A.
1778,ocds-5wno2w-SIE-GADGPA-2025-020-30745,EC-RUC-17924702930013-513704,UNION CEMENTERA NACIONAL UCEM S.A.
1985,ocds-5wno2w-SIE-AMEP-2025-006-699688,EC-RUC-17924702930013-513704,UNION CEMENTERA NACIONAL UCEM S.A.
2011,ocds-5wno2w-SIE-GAD-MM-2025-007-21655,EC-RUC-17924702930013-513704,UNION CEMENTERA NACIONAL UCEM S.A.
2880,ocds-5wno2w-SIE-GADPASTOCALLE-2025-002-115961,EC-RUC-17924702930013-513704,UNION CEMENTERA NACIONAL UCEM S.A.
2944,ocds-5wno2w-SIE-GADIPMC-2025-20-2472,EC-RUC-17924702930013-513704,UNION CEMENTERA NACIONAL UCEM S.A.
3018,ocds-5wno2w-SIE-UPEC-2025-014-24327,EC-RUC-04600365900011-848842,UPEC-CREATIVA EP
4315,ocds-5wno2w-SIE-HPDA-2025-038-2728,EC-RUC-06608379900011-533723,EMPRESA PUBLICA MUNICIPAL DE TRANSFORMACION Y ...
4942,ocds-5wno2w-SIE-HGPT-2025-041-2418,EC-RUC-17924702930013-513704,UNION CEMENTERA NACIONAL UCEM S.A.
5490,ocds-5wno2w-SIE-EPEMAPAA-2025-053-545597,EC-RUC-17924702930013-513704,UNION CEMENTERA NACIONAL UCEM S.A.


In [168]:
# Revisar los 5 proveedores con identificadores atípicos dentro de la sección parties.
# El objetivo es comprobar si SERCOP publica para esos mismos actores
# un identifier.id alternativo que permita validar el RUC sin modificarlo arbitrariamente.

supplier_ids_atipicos = set(
    awards_df.loc[
        awards_df["supplier_ruc_validado"].isna(),
        "supplier_id"
    ]
)

datos_parties_atipicos = []

for release in todos_los_releases:
    for party in release.get("parties", []):
        if party.get("id") in supplier_ids_atipicos:
            identifier = party.get("identifier", {})

            datos_parties_atipicos.append({
                "party_id": party.get("id"),
                "party_name": party.get("name"),
                "identifier_id": identifier.get("id"),
                "identifier_scheme": identifier.get("scheme"),
                "legal_name": identifier.get("legalName")
            })

pd.DataFrame(datos_parties_atipicos).drop_duplicates()

,party_id,party_name,identifier_id,identifier_scheme,legal_name
0,EC-RUC-17924702930013-513704,UNION CEMENTERA NACIONAL UCEM S.A.,EC-RUC-17924702930013-513704,EC-RUC,UNION CEMENTERA NACIONAL UCEM S.A.
5,EC-RUC-01903785860011-574645,Compañia de Economia Mixta Agroazuay GPA,EC-RUC-01903785860011-574645,EC-RUC,Compañia de Economia Mixta Agroazuay GPA
6,EC-RUC-06608379900011-533723,EMPRESA PUBLICA MUNICIPAL DE TRANSFORMACION Y ...,EC-RUC-06608379900011-533723,EC-RUC,EMPRESA PUBLICA MUNICIPAL DE TRANSFORMACION Y ...
9,EC-RUC-04600365900011-848842,UPEC-CREATIVA EP,EC-RUC-04600365900011-848842,EC-RUC,UPEC-CREATIVA EP
32,EC-RUC-17681525600011-236022,CORPORACION NACIONAL DE TELECOMUNICACIONES,EC-RUC-17681525600011-236022,EC-RUC,CORPORACION NACIONAL DE TELECOMUNICACIONES


In [169]:
# Extraer el valor referencial desde el único lote disponible en cada procedimiento.
# Se conserva como nulo cuando el procedimiento no contiene lotes,
# evitando imputar valores que no están publicados por SERCOP.

valores_referenciales = []

for release in todos_los_releases:
    tender = release.get("tender", {})
    lots = tender.get("lots", [])

    if len(lots) > 0:
        value = lots[0].get("value", {})
        tender_value = value.get("amount")
        tender_currency = value.get("currency")
        tender_value_source = "tender.lots[0].value"
    else:
        tender_value = None
        tender_currency = None
        tender_value_source = None

    valores_referenciales.append({
        "ocid": release.get("ocid"),
        "tender_value": tender_value,
        "tender_currency": tender_currency,
        "tender_value_source": tender_value_source
    })

tender_values_df = pd.DataFrame(valores_referenciales)

tender_values_df.isnull().sum()


ocid                   0
tender_value           1
tender_currency        1
tender_value_source    1
dtype: int64

In [170]:
# Verificar la moneda de los valores referenciales disponibles.
# El procedimiento sin lote permanece como nulo y no se imputa.

tender_values_df["tender_currency"].value_counts(dropna=False)

tender_currency
USD    18325
NaN        1
Name: count, dtype: int64

In [171]:
# Comprobar si existen valores referenciales iguales a cero o negativos.
# Esto es necesario antes de aplicar transformaciones logarítmicas
# y para detectar posibles inconsistencias monetarias.

tender_valores_cero = (tender_values_df["tender_value"] == 0).sum()
tender_valores_negativos = (tender_values_df["tender_value"] < 0).sum()

tender_valores_cero, tender_valores_negativos

(np.int64(0), np.int64(0))

In [172]:
# Revisar cuántos ítems contiene cada adjudicación.
# Esto permite saber si una adjudicación puede estar asociada a uno o varios CPC
# antes de construir la variable de objeto contractual para el análisis de recurrencia.

cantidad_items_award = []

for release in todos_los_releases:
    for award in release.get("awards", []):
        cantidad_items_award.append(
            len(award.get("items", []))
        )

pd.Series(cantidad_items_award).value_counts().sort_index()

1    15718
Name: count, dtype: int64

In [173]:
# Extraer el CPC principal de cada adjudicación.
# Como todas las adjudicaciones contienen exactamente un ítem,
# se toma items[0].classification como clasificación principal.
# También se conserva la descripción original para facilitar la interpretación.

filas_cpc = []

for release in todos_los_releases:
    ocid = release.get("ocid")

    for award in release.get("awards", []):
        items = award.get("items", [])

        if items:
            clasificacion = items[0].get("classification", {})

            cpc_id = clasificacion.get("id")
            cpc_scheme = clasificacion.get("scheme")
            cpc_description = clasificacion.get("description")
        else:
            cpc_id = None
            cpc_scheme = None
            cpc_description = None

        filas_cpc.append({
            "ocid": ocid,
            "award_id": award.get("id"),
            "cpc_id": cpc_id,
            "cpc_scheme": cpc_scheme,
            "cpc_description": cpc_description
        })

cpc_df = pd.DataFrame(filas_cpc)

cpc_df.isnull().sum()

ocid               0
award_id           0
cpc_id             0
cpc_scheme         0
cpc_description    0
dtype: int64

In [174]:
# Crear el CPC a nivel de 5 dígitos para el análisis de recurrencia.
# Se conservan tanto el CPC original como su versión agregada,
# de manera que no se pierda el detalle publicado por SERCOP.

cpc_df["cpc_5"] = (
    cpc_df["cpc_id"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .str[:5]
)

# Revisar algunos ejemplos para comprobar la transformación

cpc_df[
    ["cpc_id", "cpc_5", "cpc_description"]
].head(10)

,cpc_id,cpc_5,cpc_description
0,3544002144,35440,RESINA LIQUIDA
1,35260712432,35260,"Ibuprofeno, Sólido oral, 400 mg, Caja x blíste..."
2,3425000120,34250,CALCIO CARBONATO
3,641000023,64100,SERVICIOS DE TRANSPORTE CON CAMIONETAS DOBLE C...
4,2823612222,28236,UNIFORME DEPORTIVO
5,2113119122,21131,CANASTA DE PRODUCTOS CARNICOS Y HUEVOS
6,354400111,35440,REACTIVOS COMPUESTOS PARA DIAGNOSTICO O LABORA...
7,352901091,35290,INSUMOS DE USO GENERAL
8,352901091,35290,INSUMOS DE USO GENERAL
9,352901091,35290,INSUMOS DE USO GENERAL


In [175]:
# Validar que todos los CPC agregados tengan exactamente 5 dígitos.
# Esto asegura que la variable cpc_5 sea consistente antes de utilizarla
# para agrupar objetos contractuales en el análisis de recurrencia.

cpc_df["longitud_cpc_5"] = cpc_df["cpc_5"].str.len()

cpc_df["longitud_cpc_5"].value_counts().sort_index()

longitud_cpc_5
5    15718
Name: count, dtype: int64

In [176]:
# Integrar el CPC de 5 dígitos a la tabla de adjudicaciones
# para utilizarlo posteriormente en el análisis de recurrencia.

awards_df = awards_df.merge(
    cpc_df[
        ["ocid", "award_id", "cpc_id", "cpc_5", "cpc_description"]
    ],
    on=["ocid", "award_id"],
    how="left"
)

awards_df[
    ["ocid", "supplier_name", "cpc_id", "cpc_5"]
].head()

,ocid,supplier_name,cpc_id,cpc_5
0,ocds-5wno2w-SIE-03D02-2025-02-562910,ISIDENT-MED S.A.S.,3544002144,35440
1,ocds-5wno2w-SIE-DDSG1-2025-003-128799,LABORATORIO VIDA (LABOVIDA) S.A.,35260712432,35260
2,ocds-5wno2w-SIE-DDSG1-2025-004-128799,IDEALMEDICS S.A.S.,3425000120,34250
3,ocds-5wno2w-SIE-DIRECCIONDISTRITAL04D01TULCANM...,COMPAÑIA DE TRANSPORTE MIXTO SEÑOR DE LA BUENA...,641000023,64100
4,ocds-5wno2w-SIE-GADMCD-2025-001-94742,RODAS MOSCOSO DAVE RODOLFO,2823612222,28236


In [177]:
# Verificar que la integración del CPC no haya duplicado adjudicaciones
# y que la tabla mantenga exactamente una fila por adjudicación-proveedor.

awards_df.shape

(15718, 17)

In [178]:
# Incorporar la entidad compradora a cada adjudicación.
# Esto permitirá analizar posteriormente la recurrencia entre
# comprador, proveedor y objeto contractual CPC a 5 dígitos.

awards_df = awards_df.merge(
    procedures_df[
        ["ocid", "buyer_id", "buyer_name", "buyer_name_normalized"]
    ],
    on="ocid",
    how="left"
)

awards_df[
    ["ocid", "buyer_name", "supplier_name", "cpc_5", "award_date"]
].head()

,ocid,buyer_name,supplier_name,cpc_5,award_date
0,ocds-5wno2w-SIE-03D02-2025-02-562910,DIRECCION DISTRITAL 03D02 CAÑAR-EL TAMBO- SUSC...,ISIDENT-MED S.A.S.,35440,2025-04-01T13:45:15-05:00
1,ocds-5wno2w-SIE-DDSG1-2025-003-128799,DIRECCION DISTRITAL DE SALUD # 09D01,LABORATORIO VIDA (LABOVIDA) S.A.,35260,2025-03-31T17:28:27-05:00
2,ocds-5wno2w-SIE-DDSG1-2025-004-128799,DIRECCION DISTRITAL DE SALUD # 09D01,IDEALMEDICS S.A.S.,34250,2025-04-02T11:16:07-05:00
3,ocds-5wno2w-SIE-DIRECCIONDISTRITAL04D01TULCANM...,DIRECCION DISTRITAL 04D01-SAN PEDRO DE HUACA-T...,COMPAÑIA DE TRANSPORTE MIXTO SEÑOR DE LA BUENA...,64100,2025-03-27T16:03:47-05:00
4,ocds-5wno2w-SIE-GADMCD-2025-001-94742,GOBIERNO AUTONOMO DESCENTRALIZADO MUNICIPAL DE...,RODAS MOSCOSO DAVE RODOLFO,28236,2025-03-27T10:08:43-05:00


In [179]:
# Verificar que la incorporación de la entidad compradora
# no haya generado duplicados en la tabla de adjudicaciones.

awards_df.shape

(15718, 20)

In [180]:
# Verificar que las variables necesarias para el análisis de recurrencia
# estén completas: comprador, proveedor, CPC a 5 dígitos y fecha de adjudicación.

awards_df[
    ["buyer_id", "supplier_id", "cpc_5", "award_date"]
].isnull().sum()

buyer_id       0
supplier_id    0
cpc_5          0
award_date     0
dtype: int64

In [181]:
# Verificar que los procedimientos analizados correspondan únicamente
# a Subasta Inversa Electrónica y que no existan registros de Catálogo Electrónico.
# Esta validación responde directamente a la corrección metodológica del profesor.

modalidades = []

for release in todos_los_releases:
    tender = release.get("tender", {})

    modalidades.append({
        "procurement_method": tender.get("procurementMethod"),
        "procurement_method_details": tender.get("procurementMethodDetails")
    })

modalidades_df = pd.DataFrame(modalidades)

modalidades_df["procurement_method_details"].value_counts(dropna=False)

procurement_method_details
Subasta Inversa Electrónica    18326
Name: count, dtype: int64

In [182]:
# Preparar la fecha local de adjudicación para el análisis de recurrencia.
# Se conserva la fecha y hora local publicada por SERCOP, sin convertirla a UTC,
# porque la recurrencia se evaluará dentro de ventanas temporales de 90 días.

awards_df["award_date_local"] = pd.to_datetime(
    awards_df["award_date"].str[:19],
    errors="coerce"
)

print(
    "Fechas de adjudicación nulas después de la conversión:",
    awards_df["award_date_local"].isna().sum()
)

print(
    "Fecha mínima:",
    awards_df["award_date_local"].min()
)

print(
    "Fecha máxima:",
    awards_df["award_date_local"].max()
)

Fechas de adjudicación nulas después de la conversión: 0
Fecha mínima: 2025-01-30 09:52:37
Fecha máxima: 2026-07-06 10:18:48


In [183]:
# Ordenar las adjudicaciones por comprador, proveedor, CPC a 5 dígitos y fecha.
# Esta estructura permitirá calcular posteriormente cuántos procedimientos
# del mismo vínculo comprador-proveedor-objeto ocurren dentro de una ventana de 90 días.

awards_recurrencia_df = awards_df.sort_values(
    by=[
        "buyer_id",
        "supplier_id",
        "cpc_5",
        "award_date_local"
    ]
).copy()

awards_recurrencia_df[
    [
        "buyer_name",
        "supplier_name",
        "cpc_5",
        "award_date_local"
    ]
].head(10)

,buyer_name,supplier_name,cpc_5,award_date_local
116,GOBIERNO AUTÓNOMO DESCENTRALIZADO PROVINCIAL D...,MOSCOSO GAVILANES PATRICIO EDMUNDO,15320,2025-04-25 11:44:41
11601,GOBIERNO AUTÓNOMO DESCENTRALIZADO PROVINCIAL D...,MOSCOSO GAVILANES PATRICIO EDMUNDO,15320,2025-09-04 15:48:16
6319,GOBIERNO AUTÓNOMO DESCENTRALIZADO PROVINCIAL D...,CORDERO GARATE MILTON ALFONSO,54800,2026-03-11 15:26:21
14105,GOBIERNO AUTÓNOMO DESCENTRALIZADO PROVINCIAL D...,PAREDEZ SAAVEDRA CLEVER ARTURO,87340,2025-08-27 16:34:45
15557,GOBIERNO AUTÓNOMO DESCENTRALIZADO PROVINCIAL D...,VILLALTA NUGRA DIANA REBECA,96220,2025-09-22 14:00:02
11637,GOBIERNO AUTÓNOMO DESCENTRALIZADO PROVINCIAL D...,NARVAEZ TERREROS BLANCA NOEMI,48170,2025-07-21 16:20:24
12079,GOBIERNO AUTÓNOMO DESCENTRALIZADO PROVINCIAL D...,ZEAS TAPIA MONICA CATALINA,27160,2025-07-04 14:34:23
9557,GOBIERNO AUTÓNOMO DESCENTRALIZADO PROVINCIAL D...,CALDAS ALVAREZ TOMAS EDUARDO,44421,2025-06-02 15:54:45
12321,GOBIERNO AUTÓNOMO DESCENTRALIZADO PROVINCIAL D...,CASTRO AGUILAR JULIO CESAR,46420,2025-07-23 09:14:05
1183,GOBIERNO AUTÓNOMO DESCENTRALIZADO PROVINCIAL D...,PATIÑO VIZHCO MARIA FERNANDA,96220,2025-05-01 08:28:47


In [184]:
# Calcular recurrencia según la regla solicitada por el profesor:
# mismo comprador + mismo proveedor + mismo CPC a 5 dígitos
# con más de 3 procedimientos distintos dentro de una ventana de 90 días.
#
# Para cada combinación se calcula el máximo número de procedimientos
# observados en cualquier intervalo consecutivo de 90 días.

resultados_recurrencia = []

for (buyer_id, supplier_id, cpc_5), grupo in awards_recurrencia_df.groupby(
    ["buyer_id", "supplier_id", "cpc_5"]
):

    grupo = grupo.sort_values("award_date_local").copy()
    fechas = grupo["award_date_local"].tolist()
    ocids = grupo["ocid"].tolist()

    max_procesos_90d = 0
    fecha_inicio_max = None
    fecha_fin_max = None

    inicio = 0

    for fin in range(len(fechas)):

        while fechas[fin] - fechas[inicio] > pd.Timedelta(days=90):
            inicio += 1

        # Contar procedimientos OCID distintos dentro de la ventana
        cantidad = len(set(ocids[inicio:fin + 1]))

        if cantidad > max_procesos_90d:
            max_procesos_90d = cantidad
            fecha_inicio_max = fechas[inicio]
            fecha_fin_max = fechas[fin]

    resultados_recurrencia.append({
        "buyer_id": buyer_id,
        "supplier_id": supplier_id,
        "cpc_5": cpc_5,
        "max_procesos_90d": max_procesos_90d,
        "fecha_inicio_ventana": fecha_inicio_max,
        "fecha_fin_ventana": fecha_fin_max
    })

recurrencia_90d_df = pd.DataFrame(resultados_recurrencia)

# Casos que cumplen la regla del profesor: más de 3 procesos en 90 días
casos_recurrentes_90d = recurrencia_90d_df[
    recurrencia_90d_df["max_procesos_90d"] > 3
]

len(casos_recurrentes_90d)

23

In [185]:
# Incorporar los nombres de compradores y proveedores a los casos recurrentes
# para facilitar su interpretación y revisión.
# Los identificadores siguen siendo la clave principal; los nombres se usan solo para presentación.

buyer_names = (
    awards_df[
        ["buyer_id", "buyer_name"]
    ]
    .drop_duplicates(subset=["buyer_id"])
)

supplier_names = (
    awards_df[
        ["supplier_id", "supplier_name"]
    ]
    .drop_duplicates(subset=["supplier_id"])
)

casos_recurrentes_detalle = (
    casos_recurrentes_90d
    .merge(buyer_names, on="buyer_id", how="left")
    .merge(supplier_names, on="supplier_id", how="left")
    .sort_values(
        "max_procesos_90d",
        ascending=False
    )
)

casos_recurrentes_detalle[
    [
        "buyer_name",
        "supplier_name",
        "cpc_5",
        "max_procesos_90d",
        "fecha_inicio_ventana",
        "fecha_fin_ventana"
    ]
]

,buyer_name,supplier_name,cpc_5,max_procesos_90d,fecha_inicio_ventana,fecha_fin_ventana
1,GAD MUNICIPAL DE AZOGUES,COMISARIATO ECONOMICO COMIECON C. LTDA.,62221,5,2025-06-13 11:36:39,2025-08-28 14:28:06
5,Hospital Guayaquil Abel Gilbert Ponton,VIBAG C.A.,35290,5,2025-07-04 20:05:49,2025-08-18 17:26:30
6,HOSPITAL DE ESPECIALIDADES - TEODORO MALDONADO...,SIMED S. A.,35290,5,2025-11-07 17:04:00,2025-12-31 14:53:50
20,Hospital General del Sur de Quito,LABORATORIO VIDA (LABOVIDA) S.A.,35260,5,2025-06-05 14:28:32,2025-08-06 17:09:17
19,Hospital General del Sur de Quito,ALVAREZ LARREA EQUIPOS MEDICOS ALEM CIA. LTDA.,35290,5,2025-04-16 16:37:21,2025-05-27 15:22:58
11,HOSPITAL DE ESPECIALIDADES FUERZAS ARMADAS NO. 1,B.BRAUN MEDICAL S.A.,35290,5,2025-04-08 19:07:18,2025-05-21 09:41:16
9,GOBIERNO AUTONOMO DESCENTRALIZADO INTERCULTURA...,SALGADO MENDEZ MANUEL AURELIO,37370,5,2025-10-17 17:02:32,2025-11-21 11:53:52
3,HOSPITAL PROVINCIAL GENERAL DOCENTE RIOBAMBA,PRODUCTOS Y DISTRIBUCIONES MEDICAS ANDINO PROD...,35290,4,2025-05-21 20:45:13,2025-06-11 09:14:28
4,Municipalidad de Guayaquil,KRONOS LABORATORIOS C. LTDA.,35260,4,2025-06-19 17:13:44,2025-09-03 12:42:11
0,HOSPITAL DE ESPECIALIDADES JOSÉ CARRASCO ARTEAGA,CORPOMEDICA CIA. LTDA.,35290,4,2025-04-22 11:10:02,2025-05-29 16:37:22


In [186]:
# Revisar la intensidad de los 23 casos recurrentes.
# Se cuenta cuántas relaciones presentan 4, 5, 6 o más procedimientos
# del mismo comprador-proveedor-CPC5 dentro de una ventana de 90 días.

casos_recurrentes_detalle[
    "max_procesos_90d"
].value_counts().sort_index()

max_procesos_90d
4    16
5     7
Name: count, dtype: int64

In [187]:
# Revisar la disponibilidad del campo award.status.
# Si el estado de adjudicación no está publicado, se mantendrá como nulo
# y no se inferirá a partir de otros campos para evitar imputaciones arbitrarias.

estados_award = []

for release in todos_los_releases:
    for award in release.get("awards", []):
        estados_award.append({
            "ocid": release.get("ocid"),
            "award_id": award.get("id"),
            "award_status": award.get("status")
        })

award_status_df = pd.DataFrame(estados_award)

print(
    "Adjudicaciones con award.status nulo:",
    award_status_df["award_status"].isna().sum()
)

award_status_df["award_status"].value_counts(dropna=False)

Adjudicaciones con award.status nulo: 15718


award_status
None    15718
Name: count, dtype: int64

In [188]:
# Revisar la disponibilidad de tenderPeriod.endDate.
# Cuando la fecha final del período de licitación no esté publicada,
# se mantendrá como nula y no se sustituirá por otra fecha OCDS.

fechas_fin_tender = []

for release in todos_los_releases:
    tender = release.get("tender", {})
    tender_period = tender.get("tenderPeriod", {})

    fechas_fin_tender.append({
        "ocid": release.get("ocid"),
        "tender_end_date": tender_period.get("endDate")
    })

tender_end_df = pd.DataFrame(fechas_fin_tender)

print(
    "Procedimientos con tenderPeriod.endDate nulo:",
    tender_end_df["tender_end_date"].isna().sum()
)

tender_end_df["tender_end_date"].isna().value_counts()

Procedimientos con tenderPeriod.endDate nulo: 18326


tender_end_date
True    18326
Name: count, dtype: int64

In [189]:
# Validar la consistencia entre tender.numberOfTenderers
# y la cantidad real de oferentes registrados en tender.tenderers.
# Los casos donde numberOfTenderers es nulo se mantienen aparte
# y no se interpretan como cero oferentes.

comparacion_oferentes = []

for release in todos_los_releases:
    tender = release.get("tender", {})

    numero_reportado = tender.get("numberOfTenderers")
    cantidad_observada = len(tender.get("tenderers", []))

    comparacion_oferentes.append({
        "ocid": release.get("ocid"),
        "numero_reportado": numero_reportado,
        "cantidad_observada": cantidad_observada
    })

comparacion_oferentes_df = pd.DataFrame(comparacion_oferentes)

casos_comparables = comparacion_oferentes_df[
    comparacion_oferentes_df["numero_reportado"].notna()
]

diferencias_oferentes = casos_comparables[
    casos_comparables["numero_reportado"] !=
    casos_comparables["cantidad_observada"]
]

print("Procedimientos comparables:", len(casos_comparables))
print("Procedimientos con diferencias:", len(diferencias_oferentes))

Procedimientos comparables: 18033
Procedimientos con diferencias: 0


In [190]:
# Identificar valores extremos en el monto adjudicado mediante la regla de 1.5 × IQR.
# Estos registros NO se eliminan automáticamente, porque montos elevados pueden
# corresponder a contrataciones legítimas. La detección se utiliza para documentar
# la asimetría de los datos y definir posteriormente una visualización adecuada.

Q1 = awards_df["award_amount"].quantile(0.25)
Q3 = awards_df["award_amount"].quantile(0.75)

IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_award = awards_df[
    (awards_df["award_amount"] < limite_inferior) |
    (awards_df["award_amount"] > limite_superior)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Límite superior:", limite_superior)
print("Valores extremos identificados:", len(outliers_award))

Q1: 16647.057500000003
Q3: 79832.0475
IQR: 63184.99
Límite superior: 174609.5325
Valores extremos identificados: 1831


In [191]:
# Cuantificar la presencia de valores extremos y revisar el monto máximo.
# Los valores originales se conservan para los cálculos; esta revisión
# servirá para justificar transformaciones únicamente de visualización.

porcentaje_outliers = len(outliers_award) / len(awards_df) * 100
monto_maximo = awards_df["award_amount"].max()

print("Porcentaje de valores extremos:", round(porcentaje_outliers, 2), "%")
print("Monto adjudicado máximo:", monto_maximo)

Porcentaje de valores extremos: 11.65 %
Monto adjudicado máximo: 12840063.97


In [192]:
# Calcular el percentil 99 del monto adjudicado.
# Este valor se utilizará posteriormente como referencia para una posible
# winsorización exclusiva de las visualizaciones.
# Los montos originales NO se modifican para los cálculos estadísticos.

percentil_99 = awards_df["award_amount"].quantile(0.99)

casos_sobre_p99 = (
    awards_df["award_amount"] > percentil_99
).sum()

print("Percentil 99:", percentil_99)
print("Adjudicaciones por encima del percentil 99:", casos_sobre_p99)

Percentil 99: 1006458.1959999994
Adjudicaciones por encima del percentil 99: 158


In [193]:
# Crear variables auxiliares exclusivamente para visualización.
# Los montos adjudicados originales se mantienen sin modificación.
# log10 permite representar adecuadamente la fuerte asimetría monetaria,
# mientras que la winsorización al percentil 99 evita que unos pocos
# valores extremos dominen visualmente las gráficas.

import numpy as np

awards_df["award_amount_log10"] = np.log10(
    awards_df["award_amount"]
)

awards_df["award_amount_winsor_p99"] = (
    awards_df["award_amount"].clip(upper=percentil_99)
)

awards_df[
    [
        "award_amount",
        "award_amount_log10",
        "award_amount_winsor_p99"
    ]
].describe()

,award_amount,award_amount_log10,award_amount_winsor_p99
count,1.571800e+04,15718.000000,1.571800e+04
mean,9.575976e+04,4.598415,8.419583e+04
std,2.825150e+05,0.500717,1.470415e+05
min,1.754820e+03,3.244233,1.754820e+03
25%,1.664706e+04,4.221337,1.664706e+04
50%,3.260126e+04,4.513234,3.260126e+04
75%,7.983205e+04,4.902177,7.983205e+04
max,1.284006e+07,7.108567,1.006458e+06


In [194]:
# Detectar nombres registrados como texto "null", "none" o vacío.
# Estos valores pueden no ser reconocidos automáticamente como nulos por Python,
# por lo que se revisan antes de cerrar la fase de limpieza.

def contar_textos_nulos(serie):
    return (
        serie.astype(str)
        .str.strip()
        .str.lower()
        .isin(["null", "none", ""])
        .sum()
    )

print(
    "Buyer names problemáticos:",
    contar_textos_nulos(procedures_df["buyer_name"])
)

print(
    "Tenderer names problemáticos:",
    contar_textos_nulos(tender_participation_df["tenderer_name"])
)

print(
    "Supplier names problemáticos:",
    contar_textos_nulos(awards_df["supplier_name"])
)

Buyer names problemáticos: 0
Tenderer names problemáticos: 27
Supplier names problemáticos: 9


In [195]:
# Revisar los oferentes y proveedores cuyos nombres aparecen como
# texto nulo, vacío o "none".
# El objetivo es identificar sus IDs antes de buscar si SERCOP
# publica un nombre válido para esos mismos actores en la sección parties.

valores_nulos_texto = ["null", "none", ""]

oferentes_nombre_problematico = tender_participation_df[
    tender_participation_df["tenderer_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(valores_nulos_texto)
][["ocid", "tenderer_id", "tenderer_name"]]

proveedores_nombre_problematico = awards_df[
    awards_df["supplier_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(valores_nulos_texto)
][["ocid", "supplier_id", "supplier_name"]]

print("Oferentes problemáticos:")
display(oferentes_nombre_problematico)

print("Proveedores adjudicados problemáticos:")
display(proveedores_nombre_problematico)

Oferentes problemáticos:


,ocid,tenderer_id,tenderer_name
12671,ocds-5wno2w-SIE-GADM-CH-2025-004-136206,EC-RUC-1891750028001-451881,null
21693,ocds-5wno2w-SIE-GADMCPZ-2025-058-2345,EC-RUC-1691715422001-835773,null
21981,ocds-5wno2w-SIE-GADPO-2025-134-29112,EC-RUC-1792693209001-772428,null
26395,ocds-5wno2w-SIE-GADMCO-2025-038-26427,EC-RUC-1891750028001-451881,null
27542,ocds-5wno2w-SIE-GADMLA-2025-097-91040,EC-RUC-1891750028001-451881,null
28745,ocds-5wno2w-SIE-EPP-2025-288-253178,EC-RUC-1792041694001-42946,null
29182,ocds-5wno2w-SIE-RPMO-2025-003-382389,EC-RUC-1891750028001-451881,null
29217,ocds-5wno2w-SIE-GADMLL-2025-004-74677,EC-RUC-0790152018001-14282,null
30809,ocds-5wno2w-SIE-UNEMI-2025-663-43664,EC-RUC-1891750028001-451881,null
31914,ocds-5wno2w-SIE-GADQ-2025-00003-88963,EC-RUC-1891750028001-451881,null


Proveedores adjudicados problemáticos:


,ocid,supplier_id,supplier_name
3999,ocds-5wno2w-SIE-GADPO-2025-134-29112,EC-RUC-1792693209001-772428,null
5521,ocds-5wno2w-SIE-RPMO-2025-003-382389,EC-RUC-1891750028001-451881,null
7549,ocds-5wno2w-SIE-PPSSPZ-2025-012-385338,EC-RUC-1691715422001-835773,null
9119,ocds-5wno2w-SIE-EMAM-2025-001-302595,EC-RUC-0790152018001-14282,null
12021,ocds-5wno2w-SIE-DDS16D02-2025-00005-583950,EC-RUC-1691715422001-835773,null
13998,ocds-5wno2w-SIE-EPP-2025-035-253178,EC-RUC-1792693209001-772428,null
14778,ocds-5wno2w-SIE-MPORTO-2025-028-2332,EC-RUC-1391898320001-934023,null
15342,ocds-5wno2w-SIE-DDS16D01-2025-0012-44928,EC-RUC-1691715422001-835773,null
15479,ocds-5wno2w-SIE-CZ2-2025-009-457506,EC-RUC-1691715422001-835773,null


In [196]:
# Buscar en parties un nombre válido para los oferentes cuyo tenderer_name aparece como "null".
# Se utiliza el mismo tenderer_id como clave para recuperar el nombre publicado
# en otra parte del registro OCDS, sin hacer emparejamientos aproximados.

ids_oferentes_problematicos = set(
    oferentes_nombre_problematico["tenderer_id"]
)

nombres_parties_oferentes = []

for release in todos_los_releases:
    for party in release.get("parties", []):
        if party.get("id") in ids_oferentes_problematicos:
            nombre = party.get("name")

            nombres_parties_oferentes.append({
                "tenderer_id": party.get("id"),
                "nombre_parties": nombre
            })

nombres_parties_oferentes_df = (
    pd.DataFrame(nombres_parties_oferentes)
    .drop_duplicates()
)

print(
    "IDs problemáticos únicos:",
    oferentes_nombre_problematico["tenderer_id"].nunique()
)

nombres_parties_oferentes_df

IDs problemáticos únicos: 8


,tenderer_id,nombre_parties
0,EC-RUC-1891750028001-451881,FULL TECNOLOGIA FULLTEC CIA LTDA
2,EC-RUC-1891750028001-451881,FULL TECNOLOGIA FULLTEC CIA. LTDA.
3,EC-RUC-1792693209001-772428,CONSTRUCTORAJANOVIC S.A.
9,EC-RUC-0790152018001-14282,CONSTRUCTORA ANDRES SOLANO ANDRESOL S A
10,EC-RUC-1891750028001-451881,null
19,EC-RUC-1691715422001-835773,null
20,EC-RUC-1792693209001-772428,null
37,EC-RUC-1792041694001-42946,null
38,EC-RUC-1792041694001-42946,LISERPE S.A.
40,EC-RUC-0790152018001-14282,null


In [197]:
# Revisar si cada oferente con nombre problemático posee al menos un nombre válido en parties.
# Se excluyen los textos "null", "none" y vacíos, y se normalizan los nombres
# para evitar contar como diferentes variaciones de mayúsculas, tildes o espacios.

nombres_parties_validos = nombres_parties_oferentes_df[
    ~nombres_parties_oferentes_df["nombre_parties"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["null", "none", ""])
].copy()

nombres_parties_validos["nombre_normalizado"] = (
    nombres_parties_validos["nombre_parties"]
    .apply(normalizar_nombre)
)

resumen_nombres_oferentes = (
    nombres_parties_validos
    .groupby("tenderer_id")
    .agg(
        nombres_validos=("nombre_parties", "count"),
        nombres_normalizados_distintos=("nombre_normalizado", "nunique")
    )
)

print("IDs problemáticos únicos:", oferentes_nombre_problematico["tenderer_id"].nunique())
print("IDs con al menos un nombre válido en parties:", resumen_nombres_oferentes.shape[0])

resumen_nombres_oferentes

IDs problemáticos únicos: 8
IDs con al menos un nombre válido en parties: 5


,nombres_validos,nombres_normalizados_distintos
tenderer_id,,
EC-RUC-0790152018001-14282,1,1
EC-RUC-1792041694001-42946,1,1
EC-RUC-1792260043001-283853,1,1
EC-RUC-1792693209001-772428,1,1
EC-RUC-1891750028001-451881,2,2


In [198]:
# Mostrar los nombres válidos encontrados en parties para cada oferente problemático.
# Esto permite decidir de forma transparente qué nombre puede recuperarse
# y cuáles casos deben mantenerse sin nombre por falta de evidencia.

nombres_validos_por_oferente = (
    nombres_parties_validos
    .groupby("tenderer_id")["nombre_parties"]
    .unique()
)

nombres_validos_por_oferente

tenderer_id
EC-RUC-0790152018001-14282             [CONSTRUCTORA ANDRES SOLANO ANDRESOL S A]
EC-RUC-1792041694001-42946                                        [LISERPE S.A.]
EC-RUC-1792260043001-283853                      [CONSTRUCTORA LA ROCA CLR S.A.]
EC-RUC-1792693209001-772428                           [CONSTRUCTORAJANOVIC S.A.]
EC-RUC-1891750028001-451881    [FULL TECNOLOGIA FULLTEC CIA LTDA, FULL TECNOL...
Name: nombre_parties, dtype: object

In [199]:
# Buscar nombres válidos para los 8 oferentes problemáticos dentro de toda
# la tabla de participación. Si el mismo tenderer_id aparece correctamente
# nombrado en otro procedimiento, ese registro puede utilizarse como evidencia
# para recuperar el nombre sin hacer emparejamientos por similitud.

ids_oferentes_problematicos = set(
    oferentes_nombre_problematico["tenderer_id"]
)

nombres_validos_misma_tabla = tender_participation_df[
    tender_participation_df["tenderer_id"].isin(ids_oferentes_problematicos)
].copy()

nombres_validos_misma_tabla = nombres_validos_misma_tabla[
    ~nombres_validos_misma_tabla["tenderer_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["null", "none", ""])
]

nombres_validos_misma_tabla.groupby(
    "tenderer_id"
)["tenderer_name"].unique()

tenderer_id
EC-RUC-0790152018001-14282             [CONSTRUCTORA ANDRES SOLANO ANDRESOL S A]
EC-RUC-1792041694001-42946                                        [LISERPE S.A.]
EC-RUC-1792260043001-283853                      [CONSTRUCTORA LA ROCA CLR S.A.]
EC-RUC-1792693209001-772428                           [CONSTRUCTORAJANOVIC S.A.]
EC-RUC-1891750028001-451881    [FULL TECNOLOGIA FULLTEC CIA LTDA, FULL TECNOL...
Name: tenderer_name, dtype: object

In [200]:
# Identificar los oferentes problemáticos que no tienen ningún nombre válido
# recuperable ni en parties ni en otros procedimientos de la tabla de participación.
# Estos casos se conservarán con su ID y se documentarán como nombres no informados.

ids_con_nombre_valido = set(
    nombres_validos_misma_tabla["tenderer_id"]
)

ids_problematicos_totales = set(
    oferentes_nombre_problematico["tenderer_id"]
)

ids_sin_nombre_recuperable = (
    ids_problematicos_totales - ids_con_nombre_valido
)

print("IDs sin nombre recuperable:", len(ids_sin_nombre_recuperable))

oferentes_sin_nombre_recuperable = (
    oferentes_nombre_problematico[
        oferentes_nombre_problematico["tenderer_id"].isin(
            ids_sin_nombre_recuperable
        )
    ]
)

print(
    "Registros afectados:",
    len(oferentes_sin_nombre_recuperable)
)

oferentes_sin_nombre_recuperable[
    ["tenderer_id", "tenderer_name"]
].drop_duplicates()

IDs sin nombre recuperable: 3
Registros afectados: 7


,tenderer_id,tenderer_name
21693,EC-RUC-1691715422001-835773,null
69521,EC-RUC-1691728397001-1049704,null
74625,EC-RUC-1391898320001-934023,null


In [201]:
# Recuperar nombres de oferentes únicamente cuando el mismo tenderer_id
# aparece con un nombre válido en otros procedimientos.
# Se conserva la columna original y se crea una nueva columna limpia.
# Para cada ID se utiliza el nombre válido más frecuente publicado por SERCOP.
# Los casos sin evidencia suficiente se mantienen como nulos.

# Crear un diccionario con el nombre válido más frecuente por tenderer_id
nombres_validos_por_id = (
    tender_participation_df[
        ~tender_participation_df["tenderer_name"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["null", "none", ""])
    ]
    .groupby("tenderer_id")["tenderer_name"]
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

# Conservar el nombre original
tender_participation_df["nombre_oferente_limpio"] = (
    tender_participation_df["tenderer_name"]
)

# Identificar nombres problemáticos
mascara_nombre_problematico = (
    tender_participation_df["tenderer_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["null", "none", ""])
)

# Recuperar el nombre cuando exista evidencia para el mismo ID
tender_participation_df.loc[
    mascara_nombre_problematico,
    "nombre_oferente_limpio"
] = tender_participation_df.loc[
    mascara_nombre_problematico,
    "tenderer_id"
].map(nombres_validos_por_id)

# Comprobar cuántos nombres siguen sin información después de la recuperación
print(
    "Registros originalmente problemáticos:",
    mascara_nombre_problematico.sum()
)

print(
    "Registros recuperados:",
    tender_participation_df.loc[
        mascara_nombre_problematico,
        "nombre_oferente_limpio"
    ].notna().sum()
)

print(
    "Registros sin nombre recuperable:",
    tender_participation_df.loc[
        mascara_nombre_problematico,
        "nombre_oferente_limpio"
    ].isna().sum()
)

Registros originalmente problemáticos: 27
Registros recuperados: 20
Registros sin nombre recuperable: 7


In [202]:
# Revisar si los proveedores adjudicados cuyo nombre aparece como "null"
# tienen un nombre válido publicado en otras adjudicaciones con el mismo supplier_id.
# No se utiliza similitud de nombres; la recuperación se basa únicamente
# en la coincidencia exacta del identificador.

ids_proveedores_problematicos = set(
    proveedores_nombre_problematico["supplier_id"]
)

nombres_validos_proveedores = awards_df[
    awards_df["supplier_id"].isin(ids_proveedores_problematicos)
].copy()

nombres_validos_proveedores = nombres_validos_proveedores[
    ~nombres_validos_proveedores["supplier_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["null", "none", ""])
]

print(
    "IDs problemáticos únicos:",
    proveedores_nombre_problematico["supplier_id"].nunique()
)

print(
    "IDs con al menos un nombre válido:",
    nombres_validos_proveedores["supplier_id"].nunique()
)

nombres_validos_proveedores.groupby(
    "supplier_id"
)["supplier_name"].unique()

IDs problemáticos únicos: 5
IDs con al menos un nombre válido: 2


supplier_id
EC-RUC-1792693209001-772428                           [CONSTRUCTORAJANOVIC S.A.]
EC-RUC-1891750028001-451881    [FULL TECNOLOGIA FULLTEC CIA. LTDA., FULL TECN...
Name: supplier_name, dtype: object

In [203]:
# Identificar los proveedores adjudicados cuyo nombre no puede recuperarse
# desde otras adjudicaciones con el mismo supplier_id.
# Estos casos se conservarán con su identificador original y se documentarán
# como nombres no informados si no existe evidencia suficiente para completarlos.

ids_proveedores_con_nombre = set(
    nombres_validos_proveedores["supplier_id"]
)

ids_proveedores_problematicos_totales = set(
    proveedores_nombre_problematico["supplier_id"]
)

ids_proveedores_sin_nombre = (
    ids_proveedores_problematicos_totales - ids_proveedores_con_nombre
)

print(
    "IDs de proveedores sin nombre recuperable:",
    len(ids_proveedores_sin_nombre)
)

proveedores_sin_nombre_recuperable = (
    proveedores_nombre_problematico[
        proveedores_nombre_problematico["supplier_id"].isin(
            ids_proveedores_sin_nombre
        )
    ]
)

print(
    "Registros afectados:",
    len(proveedores_sin_nombre_recuperable)
)

proveedores_sin_nombre_recuperable[
    ["supplier_id", "supplier_name"]
].drop_duplicates()

IDs de proveedores sin nombre recuperable: 3
Registros afectados: 6


,supplier_id,supplier_name
7549,EC-RUC-1691715422001-835773,null
9119,EC-RUC-0790152018001-14282,null
14778,EC-RUC-1391898320001-934023,null


In [204]:
# Buscar en parties nombres válidos para los 3 proveedores adjudicados
# que no pudieron recuperarse desde otras adjudicaciones.
# La búsqueda utiliza exclusivamente la coincidencia exacta del supplier_id,
# evitando inferencias o emparejamientos aproximados.

nombres_parties_proveedores = []

for release in todos_los_releases:
    for party in release.get("parties", []):
        if party.get("id") in ids_proveedores_sin_nombre:

            nombre = party.get("name")
            identificador = party.get("identifier", {})

            nombres_parties_proveedores.append({
                "supplier_id": party.get("id"),
                "nombre_parties": nombre,
                "nombre_legal": identificador.get("legalName")
            })

nombres_parties_proveedores_df = (
    pd.DataFrame(nombres_parties_proveedores)
    .drop_duplicates()
)

nombres_parties_proveedores_df

,supplier_id,nombre_parties,nombre_legal
0,EC-RUC-0790152018001-14282,CONSTRUCTORA ANDRES SOLANO ANDRESOL S A,CONSTRUCTORA ANDRES SOLANO ANDRESOL S A
1,EC-RUC-1691715422001-835773,null,null
2,EC-RUC-0790152018001-14282,null,null
8,EC-RUC-1391898320001-934023,null,null


In [205]:
# Recuperar el nombre de los proveedores adjudicados usando únicamente
# información publicada por SERCOP para el mismo supplier_id.
#
# Prioridad:
# 1. Nombre válido observado en otras adjudicaciones.
# 2. Nombre válido encontrado en parties.
# 3. Si no existe evidencia, el nombre se mantiene como nulo.
#
# Se conserva supplier_name original para mantener trazabilidad.

# Crear columna limpia conservando inicialmente el valor original
awards_df["nombre_proveedor_limpio"] = awards_df["supplier_name"]

# Identificar los nombres originalmente problemáticos
mascara_proveedor_problematico = (
    awards_df["supplier_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["null", "none", ""])
)

# Diccionario de nombres válidos encontrados en otras adjudicaciones
nombres_validos_awards = (
    awards_df[
        ~awards_df["supplier_name"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["null", "none", ""])
    ]
    .groupby("supplier_id")["supplier_name"]
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

# Diccionario de nombres válidos encontrados en parties
parties_validos = nombres_parties_proveedores_df[
    ~nombres_parties_proveedores_df["nombre_parties"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["null", "none", ""])
]

nombres_validos_parties = (
    parties_validos
    .drop_duplicates("supplier_id")
    .set_index("supplier_id")["nombre_parties"]
    .to_dict()
)

# Primero intentar recuperar desde otras adjudicaciones
awards_df.loc[
    mascara_proveedor_problematico,
    "nombre_proveedor_limpio"
] = awards_df.loc[
    mascara_proveedor_problematico,
    "supplier_id"
].map(nombres_validos_awards)

# Para los que todavía siguen sin nombre, intentar recuperar desde parties
mascara_aun_sin_nombre = (
    mascara_proveedor_problematico &
    awards_df["nombre_proveedor_limpio"].isna()
)

awards_df.loc[
    mascara_aun_sin_nombre,
    "nombre_proveedor_limpio"
] = awards_df.loc[
    mascara_aun_sin_nombre,
    "supplier_id"
].map(nombres_validos_parties)

# Comprobar el resultado final
print(
    "Registros originalmente problemáticos:",
    mascara_proveedor_problematico.sum()
)

print(
    "Registros recuperados:",
    awards_df.loc[
        mascara_proveedor_problematico,
        "nombre_proveedor_limpio"
    ].notna().sum()
)

print(
    "Registros sin nombre recuperable:",
    awards_df.loc[
        mascara_proveedor_problematico,
        "nombre_proveedor_limpio"
    ].isna().sum()
)

print(
    "IDs únicos sin nombre recuperable:",
    awards_df.loc[
        mascara_proveedor_problematico &
        awards_df["nombre_proveedor_limpio"].isna(),
        "supplier_id"
    ].nunique()
)

Registros originalmente problemáticos: 9
Registros recuperados: 4
Registros sin nombre recuperable: 5
IDs únicos sin nombre recuperable: 2


In [206]:
# Identificar los proveedores que permanecen sin nombre después
# de aplicar todas las fuentes de recuperación disponibles.
# Se conserva su supplier_id como identificador principal.

proveedores_finales_sin_nombre = (
    awards_df.loc[
        mascara_proveedor_problematico &
        awards_df["nombre_proveedor_limpio"].isna(),
        ["supplier_id", "supplier_ruc", "ocid"]
    ]
    .drop_duplicates()
)

proveedores_finales_sin_nombre

,supplier_id,supplier_ruc,ocid
7549,EC-RUC-1691715422001-835773,1691715422001,ocds-5wno2w-SIE-PPSSPZ-2025-012-385338
12021,EC-RUC-1691715422001-835773,1691715422001,ocds-5wno2w-SIE-DDS16D02-2025-00005-583950
14778,EC-RUC-1391898320001-934023,1391898320001,ocds-5wno2w-SIE-MPORTO-2025-028-2332
15342,EC-RUC-1691715422001-835773,1691715422001,ocds-5wno2w-SIE-DDS16D01-2025-0012-44928
15479,EC-RUC-1691715422001-835773,1691715422001,ocds-5wno2w-SIE-CZ2-2025-009-457506


In [207]:
# Crear una versión normalizada del nombre limpio del proveedor.
# Se conserva supplier_name como dato original de SERCOP.
# Los proveedores sin nombre recuperable permanecen como nulos.

awards_df["nombre_proveedor_normalizado"] = (
    awards_df["nombre_proveedor_limpio"]
    .apply(normalizar_nombre)
)

print(
    "Nombres normalizados nulos:",
    awards_df["nombre_proveedor_normalizado"].isna().sum()
)

awards_df[
    [
        "supplier_id",
        "supplier_name",
        "nombre_proveedor_limpio",
        "nombre_proveedor_normalizado"
    ]
].head()

Nombres normalizados nulos: 5


,supplier_id,supplier_name,nombre_proveedor_limpio,nombre_proveedor_normalizado
0,EC-RUC-0993392435001-1223769,ISIDENT-MED S.A.S.,ISIDENT-MED S.A.S.,ISIDENT-MED S.A.S.
1,EC-RUC-0991410465001-11294,LABORATORIO VIDA (LABOVIDA) S.A.,LABORATORIO VIDA (LABOVIDA) S.A.,LABORATORIO VIDA (LABOVIDA) S.A.
2,EC-RUC-0993392648001-1220695,IDEALMEDICS S.A.S.,IDEALMEDICS S.A.S.,IDEALMEDICS S.A.S.
3,EC-RUC-0491530340001-1016001,COMPAÑIA DE TRANSPORTE MIXTO SEÑOR DE LA BUENA...,COMPAÑIA DE TRANSPORTE MIXTO SEÑOR DE LA BUENA...,COMPANIA DE TRANSPORTE MIXTO SENOR DE LA BUENA...
4,EC-RUC-0929538445001-1086792,RODAS MOSCOSO DAVE RODOLFO,RODAS MOSCOSO DAVE RODOLFO,RODAS MOSCOSO DAVE RODOLFO


In [208]:
# Comprobar cuántos nombres de proveedores permanecen sin información
# después de la limpieza y recuperación de nombres.

awards_df["nombre_proveedor_normalizado"].isna().sum()

np.int64(5)

In [209]:
# Crear una versión normalizada del nombre limpio del oferente.
# La versión normalizada se utilizará únicamente para controles y comparación;
# los nombres originales se conservan para presentación y trazabilidad.

tender_participation_df["nombre_oferente_normalizado"] = (
    tender_participation_df["nombre_oferente_limpio"]
    .apply(normalizar_nombre)
)

print(
    "Nombres de oferentes normalizados nulos:",
    tender_participation_df["nombre_oferente_normalizado"].isna().sum()
)

Nombres de oferentes normalizados nulos: 7


In [210]:
# Registrar la procedencia del nombre utilizado para cada oferente.
# Esto permite mantener trazabilidad sobre los nombres originales,
# los nombres recuperados mediante el mismo identificador y los casos
# que permanecen sin información.

tender_participation_df["fuente_nombre_oferente"] = "original"

tender_participation_df.loc[
    mascara_nombre_problematico &
    tender_participation_df["nombre_oferente_limpio"].notna(),
    "fuente_nombre_oferente"
] = "recuperado_mismo_id"

tender_participation_df.loc[
    tender_participation_df["nombre_oferente_limpio"].isna(),
    "fuente_nombre_oferente"
] = "no_informado"

tender_participation_df["fuente_nombre_oferente"].value_counts()

fuente_nombre_oferente
original               79398
recuperado_mismo_id       20
no_informado               7
Name: count, dtype: int64

In [211]:
# Registrar la procedencia del nombre utilizado para cada proveedor adjudicado.
# Esto permite distinguir los nombres originales, los recuperados mediante
# el mismo identificador y los casos que permanecen sin información.

awards_df["fuente_nombre_proveedor"] = "original"

awards_df.loc[
    mascara_proveedor_problematico &
    awards_df["nombre_proveedor_limpio"].notna(),
    "fuente_nombre_proveedor"
] = "recuperado_mismo_id"

awards_df.loc[
    awards_df["nombre_proveedor_limpio"].isna(),
    "fuente_nombre_proveedor"
] = "no_informado"

awards_df["fuente_nombre_proveedor"].value_counts()

fuente_nombre_proveedor
original               15709
no_informado               5
recuperado_mismo_id        4
Name: count, dtype: int64

In [212]:
# Realizar un control final de las tablas procesadas.
# Se verifica la cantidad de filas, duplicados en las claves principales
# y valores nulos en identificadores esenciales antes de guardar los resultados.

print("=== PROCEDIMIENTOS ===")
print("Filas:", len(procedures_df))
print("OCID duplicados:", procedures_df["ocid"].duplicated().sum())
print("OCID nulos:", procedures_df["ocid"].isna().sum())

print("\n=== PARTICIPACIÓN DE OFERENTES ===")
print("Filas:", len(tender_participation_df))
print(
    "Duplicados OCID + oferente:",
    tender_participation_df.duplicated(
        subset=["ocid", "tenderer_id"]
    ).sum()
)
print("IDs de oferente nulos:", tender_participation_df["tenderer_id"].isna().sum())

print("\n=== ADJUDICACIONES ===")
print("Filas:", len(awards_df))
print(
    "Duplicados OCID + adjudicación + proveedor:",
    awards_df.duplicated(
        subset=["ocid", "award_id", "supplier_id"]
    ).sum()
)
print("IDs de proveedor nulos:", awards_df["supplier_id"].isna().sum())
print("Montos adjudicados nulos:", awards_df["award_amount"].isna().sum())
print("CPC5 nulos:", awards_df["cpc_5"].isna().sum())

=== PROCEDIMIENTOS ===
Filas: 18326
OCID duplicados: 0
OCID nulos: 0

=== PARTICIPACIÓN DE OFERENTES ===
Filas: 79425
Duplicados OCID + oferente: 0
IDs de oferente nulos: 0

=== ADJUDICACIONES ===
Filas: 15718
Duplicados OCID + adjudicación + proveedor: 0
IDs de proveedor nulos: 0
Montos adjudicados nulos: 0
CPC5 nulos: 0


In [213]:
# Control final de la tabla de procedimientos

print("=== PROCEDIMIENTOS ===")
print("Filas:", len(procedures_df))
print("OCID duplicados:", procedures_df["ocid"].duplicated().sum())
print("OCID nulos:", procedures_df["ocid"].isna().sum())

=== PROCEDIMIENTOS ===
Filas: 18326
OCID duplicados: 0
OCID nulos: 0


In [214]:
# Incorporar a la tabla de procedimientos las variables ya validadas:
# valor referencial, moneda, fuente del valor, fecha de inicio
# y presencia de adjudicación.
# Esto deja una tabla consolidada antes de guardar los datos procesados.

# Integrar valor referencial
procedures_df = procedures_df.merge(
    tender_values_df[
        ["ocid", "tender_value", "tender_currency", "tender_value_source"]
    ],
    on="ocid",
    how="left"
)

# Integrar fecha de inicio del procedimiento
procedures_df = procedures_df.merge(
    tender_dates_df[
        ["ocid", "tender_start_date", "tender_start_year_local"]
    ],
    on="ocid",
    how="left"
)

# Crear indicador de presencia de adjudicación
ocids_con_adjudicacion = set(awards_df["ocid"])

procedures_df["tiene_adjudicacion"] = (
    procedures_df["ocid"].isin(ocids_con_adjudicacion)
)

procedures_df.shape

(18326, 18)

In [215]:
# Validar estrictamente los identificadores de los oferentes.
# Se acepta un RUC de exactamente 13 dígitos tanto para prefijos EC-RUC- como ID-.
# Esta validación permite detectar identificadores atípicos antes de cerrar
# la fase de limpieza y garantiza consistencia entre compradores,
# oferentes y proveedores adjudicados.

tender_participation_df["ruc_oferente_validado"] = (
    tender_participation_df["tenderer_id"]
    .str.extract(r"^(?:EC-RUC-|ID-)(\d{13})-")[0]
)

print(
    "Participaciones con tenderer_id sin RUC válido de 13 dígitos:",
    tender_participation_df["ruc_oferente_validado"].isna().sum()
)

print(
    "Oferentes únicos con identificador atípico:",
    tender_participation_df.loc[
        tender_participation_df["ruc_oferente_validado"].isna(),
        "tenderer_id"
    ].nunique()
)

tender_participation_df[
    tender_participation_df["ruc_oferente_validado"].isna()
][
    ["tenderer_id", "tenderer_name"]
].drop_duplicates()

Participaciones con tenderer_id sin RUC válido de 13 dígitos: 52
Oferentes únicos con identificador atípico: 12


,tenderer_id,tenderer_name
353,EC-RUC-03600339100011-757816,UNAE EP
8478,EC-RUC-17924702930013-513704,UNION CEMENTERA NACIONAL UCEM S.A.
10803,EC-RUC-17681768200011-554840,UNIVERSIDAD DE INVESTIGACIÓN DE TECNOLOGÍA EXP...
11201,EC-RUC-01903785860011-574645,Compañia de Economia Mixta Agroazuay GPA
11957,EC-RUC-06608379900011-533723,EMPRESA PUBLICA MUNICIPAL DE TRANSFORMACION Y ...
15175,EC-RUC-01600632700011-931807,Empresa Universitaria de Salud EP EUS EP
16532,EC-RUC-04600365900011-848842,UPEC-CREATIVA EP
30366,EC-RUC-17681631700011-382151,PICHINCHA COMUNICACIONES EP
35897,EC-RUC-17680143300011-59858,Dirección de Industria Aeronáutica de la FAE
35900,ID-901949124-1269375,RIOVISTA ENTERPRISE SAS


In [216]:
# Revisar en la sección parties los 12 oferentes con identificadores atípicos.
# Se consulta el identificador publicado por SERCOP, su esquema y nombre legal
# antes de decidir si deben tratarse como RUC atípicos u otro tipo de identificación.

ids_oferentes_atipicos = set(
    tender_participation_df.loc[
        tender_participation_df["ruc_oferente_validado"].isna(),
        "tenderer_id"
    ]
)

datos_oferentes_atipicos = []

for release in todos_los_releases:
    for party in release.get("parties", []):
        if party.get("id") in ids_oferentes_atipicos:

            identificador = party.get("identifier", {})

            datos_oferentes_atipicos.append({
                "tenderer_id": party.get("id"),
                "nombre": party.get("name"),
                "identifier_id": identificador.get("id"),
                "identifier_scheme": identificador.get("scheme"),
                "nombre_legal": identificador.get("legalName")
            })

oferentes_atipicos_parties_df = (
    pd.DataFrame(datos_oferentes_atipicos)
    .drop_duplicates()
)

oferentes_atipicos_parties_df

,tenderer_id,nombre,identifier_id,identifier_scheme,nombre_legal
0,EC-RUC-03600339100011-757816,UNAE EP,EC-RUC-03600339100011-757816,EC-RUC,UNAE EP
1,EC-RUC-17924702930013-513704,UNION CEMENTERA NACIONAL UCEM S.A.,EC-RUC-17924702930013-513704,EC-RUC,UNION CEMENTERA NACIONAL UCEM S.A.
4,EC-RUC-17681768200011-554840,UNIVERSIDAD DE INVESTIGACIÓN DE TECNOLOGÍA EXP...,EC-RUC-17681768200011-554840,EC-RUC,UNIVERSIDAD DE INVESTIGACIÓN DE TECNOLOGÍA EXP...
7,EC-RUC-01903785860011-574645,Compañia de Economia Mixta Agroazuay GPA,EC-RUC-01903785860011-574645,EC-RUC,Compañia de Economia Mixta Agroazuay GPA
8,EC-RUC-06608379900011-533723,EMPRESA PUBLICA MUNICIPAL DE TRANSFORMACION Y ...,EC-RUC-06608379900011-533723,EC-RUC,EMPRESA PUBLICA MUNICIPAL DE TRANSFORMACION Y ...
9,EC-RUC-01600632700011-931807,Empresa Universitaria de Salud EP EUS EP,EC-RUC-01600632700011-931807,EC-RUC,Empresa Universitaria de Salud EP EUS EP
12,EC-RUC-04600365900011-848842,UPEC-CREATIVA EP,EC-RUC-04600365900011-848842,EC-RUC,UPEC-CREATIVA EP
23,EC-RUC-17681631700011-382151,PICHINCHA COMUNICACIONES EP,EC-RUC-17681631700011-382151,EC-RUC,PICHINCHA COMUNICACIONES EP
27,EC-RUC-17680143300011-59858,Dirección de Industria Aeronáutica de la FAE,EC-RUC-17680143300011-59858,EC-RUC,Dirección de Industria Aeronáutica de la FAE
28,ID-901949124-1269375,RIOVISTA ENTERPRISE SAS,ID-901949124-1269375,NaN,RIOVISTA ENTERPRISE SAS


In [217]:
# Comprobar si los oferentes con identificadores EC-RUC atípicos de 14 dígitos
# aparecen en otros registros con una versión de 13 dígitos.
# No se modifica ningún identificador; únicamente se busca evidencia interna
# que permita determinar si existe una variante válida del mismo RUC.

# Obtener todos los identificadores únicos de oferentes
todos_ids_oferentes = set(
    tender_participation_df["tenderer_id"].dropna().unique()
)

revision_ruc_atipicos = []

for tenderer_id in ids_oferentes_atipicos:

    # Buscar la parte numérica después de EC-RUC-
    coincidencia = re.match(
        r"^EC-RUC-(\d+)-",
        str(tenderer_id)
    )

    if coincidencia:
        numero = coincidencia.group(1)

        # Solo revisar los casos de 14 dígitos
        if len(numero) == 14:
            posible_ruc_13 = numero[:13]

            coincidencias_13 = [
                actor_id
                for actor_id in todos_ids_oferentes
                if f"EC-RUC-{posible_ruc_13}-" in actor_id
            ]

            revision_ruc_atipicos.append({
                "tenderer_id_atipico": tenderer_id,
                "numero_digitos": len(numero),
                "posible_ruc_13": posible_ruc_13,
                "coincidencias_en_base": coincidencias_13
            })

revision_ruc_atipicos_df = pd.DataFrame(revision_ruc_atipicos)

revision_ruc_atipicos_df

,tenderer_id_atipico,numero_digitos,posible_ruc_13,coincidencias_en_base
0,EC-RUC-09901490540011-210259,14,0990149054001,[]
1,EC-RUC-17681631700011-382151,14,1768163170001,[]
2,EC-RUC-01903785860011-574645,14,0190378586001,[]
3,EC-RUC-17680143300011-59858,14,1768014330001,[]
4,EC-RUC-17924702930013-513704,14,1792470293001,[]
5,EC-RUC-03600339100011-757816,14,0360033910001,[]
6,EC-RUC-06608379900011-533723,14,0660837990001,[]
7,EC-RUC-17681768200011-554840,14,1768176820001,[]
8,EC-RUC-01600632700011-931807,14,0160063270001,[]
9,EC-RUC-17681525600011-236022,14,1768152560001,[]


In [218]:
# Revisar el identificador atípico de RIOVISTA ENTERPRISE SAS.
# Se verifica si el mismo actor aparece en otros registros con otro identificador
# antes de decidir cómo clasificarlo en la base procesada.

nombre_busqueda = "RIOVISTA ENTERPRISE SAS"

coincidencias_riovista = []

for release in todos_los_releases:
    for party in release.get("parties", []):
        nombre = str(party.get("name", "")).strip().upper()

        if nombre == nombre_busqueda:
            identificador = party.get("identifier", {})

            coincidencias_riovista.append({
                "party_id": party.get("id"),
                "nombre": party.get("name"),
                "identifier_id": identificador.get("id"),
                "identifier_scheme": identificador.get("scheme")
            })

pd.DataFrame(coincidencias_riovista).drop_duplicates()

,party_id,nombre,identifier_id,identifier_scheme
0,ID-901949124-1269375,RIOVISTA ENTERPRISE SAS,ID-901949124-1269375,None


In [219]:
# Validar la consistencia temporal entre el inicio del procedimiento
# y la fecha de adjudicación.
# Una adjudicación debería ocurrir en la misma fecha o después
# del inicio del período de licitación.
# Las fechas se comparan respetando la hora local publicada por SERCOP.

# Crear fecha local de inicio del procedimiento
tender_dates_df["fecha_inicio_local"] = pd.to_datetime(
    tender_dates_df["tender_start_date"].str[:19],
    errors="coerce"
)

# Incorporar la fecha de inicio a las adjudicaciones
awards_temporal_df = awards_df.merge(
    tender_dates_df[
        ["ocid", "fecha_inicio_local"]
    ],
    on="ocid",
    how="left"
)

# Identificar adjudicaciones anteriores al inicio del procedimiento
adjudicaciones_antes_inicio = awards_temporal_df[
    awards_temporal_df["award_date_local"] <
    awards_temporal_df["fecha_inicio_local"]
]

print(
    "Adjudicaciones con fecha anterior al inicio del procedimiento:",
    len(adjudicaciones_antes_inicio)
)

adjudicaciones_antes_inicio[
    [
        "ocid",
        "fecha_inicio_local",
        "award_date_local",
        "buyer_name",
        "supplier_id"
    ]
].head(20)

Adjudicaciones con fecha anterior al inicio del procedimiento: 0


,ocid,fecha_inicio_local,award_date_local,buyer_name,supplier_id


In [220]:
# Validar la consistencia entre buyer.id y tender.procuringEntity.id.
# Ambos campos deberían identificar a la entidad responsable del procedimiento.
# Esta comprobación permite detectar posibles discrepancias institucionales
# antes de cerrar la fase de limpieza.

comparacion_comprador = []

for release in todos_los_releases:
    buyer = release.get("buyer", {})
    tender = release.get("tender", {})
    procuring_entity = tender.get("procuringEntity", {})

    comparacion_comprador.append({
        "ocid": release.get("ocid"),
        "buyer_id": buyer.get("id"),
        "buyer_name": buyer.get("name"),
        "procuring_entity_id": procuring_entity.get("id"),
        "procuring_entity_name": procuring_entity.get("name")
    })

comparacion_comprador_df = pd.DataFrame(comparacion_comprador)

print(
    "ProcuringEntity ID nulos:",
    comparacion_comprador_df["procuring_entity_id"].isna().sum()
)

diferencias_comprador = comparacion_comprador_df[
    comparacion_comprador_df["buyer_id"] !=
    comparacion_comprador_df["procuring_entity_id"]
]

print(
    "Procedimientos con buyer.id distinto de procuringEntity.id:",
    len(diferencias_comprador)
)

diferencias_comprador.head(20)

ProcuringEntity ID nulos: 0
Procedimientos con buyer.id distinto de procuringEntity.id: 0


,ocid,buyer_id,buyer_name,procuring_entity_id,procuring_entity_name


In [221]:
# Generar un perfil final de las variables numéricas principales.
# Se revisan cantidad de observaciones, nulos, mínimo, máximo,
# media y mediana para documentar la calidad y naturaleza de los datos.

perfil_numerico = pd.DataFrame({
    "numero_oferentes": procedures_df["number_of_tenderers"],
    "valor_referencial": procedures_df["tender_value"],
})

# El monto adjudicado tiene granularidad de adjudicación,
# por lo que se resume por separado para no mezclar niveles de análisis.

print("=== PROCEDIMIENTOS ===")
display(
    perfil_numerico.describe().T
)

print("\nValores nulos:")
display(
    perfil_numerico.isna().sum()
)

print("\n=== ADJUDICACIONES ===")
display(
    awards_df[["award_amount"]].describe().T
)

print("\nMontos adjudicados nulos:")
print(
    awards_df["award_amount"].isna().sum()
)

=== PROCEDIMIENTOS ===


,count,mean,std,min,25%,50%,75%,max
numero_oferentes,18033.0,4.404425,3.765014,1.00,2.0,3.0,6.00000,44.0
valor_referencial,18325.0,117150.537534,380949.125459,7213.03,19960.0,39500.3,96536.12669,15996328.0



Valores nulos:


numero_oferentes     293
valor_referencial      1
dtype: int64


=== ADJUDICACIONES ===


,count,mean,std,min,25%,50%,75%,max
award_amount,15718.0,95759.758558,282514.968441,1754.82,16647.0575,32601.265,79832.0475,12840063.97



Montos adjudicados nulos:
0


In [222]:
# Marcar explícitamente el procedimiento identificado como registro atípico de calidad.
# El registro se conserva en la tabla general para mantener trazabilidad,
# pero podrá excluirse de análisis que requieran valor referencial o adjudicación,
# ya que no contiene lote, no tiene adjudicación, presenta un identificador atípico
# y su descripción corresponde a "PRUEBA".

ocid_registro_prueba = "ocds-5wno2w-SIE-CNELM-017A-2011-124705"

procedures_df["registro_atipico_calidad"] = (
    procedures_df["ocid"] == ocid_registro_prueba
)

print(
    "Registros marcados como atípicos:",
    procedures_df["registro_atipico_calidad"].sum()
)

procedures_df.loc[
    procedures_df["registro_atipico_calidad"],
    [
        "ocid",
        "buyer_id",
        "buyer_name",
        "tender_id",
        "tender_status",
        "tender_value",
        "tiene_adjudicacion"
    ]
]

Registros marcados como atípicos: 1


,ocid,buyer_id,buyer_name,tender_id,tender_status,tender_value,tiene_adjudicacion
2569,ocds-5wno2w-SIE-CNELM-017A-2011-124705,EC-RUC-09925984680012-124705,CNELMANABI,SIE-CNELM-017A-2011-124705,active,NaN,False


In [223]:
# Crear indicadores explícitos de calidad para compradores, oferentes y proveedores.
# Estas variables permiten conservar todos los registros y, al mismo tiempo,
# identificar aquellos cuyos RUC no cumplen el patrón estándar de 13 dígitos.
# No se corrigen ni eliminan identificadores sin evidencia documental.

# Entidades contratantes
procedures_df["identificador_comprador_atipico"] = (
    procedures_df["buyer_ruc_valido"].isna()
)

# Oferentes
tender_participation_df["identificador_oferente_atipico"] = (
    tender_participation_df["ruc_oferente_validado"].isna()
)

# Proveedores adjudicados
awards_df["identificador_proveedor_atipico"] = (
    awards_df["supplier_ruc_validado"].isna()
)

print(
    "Procedimientos con identificador de comprador atípico:",
    procedures_df["identificador_comprador_atipico"].sum()
)

print(
    "Participaciones con identificador de oferente atípico:",
    tender_participation_df["identificador_oferente_atipico"].sum()
)

print(
    "Adjudicaciones con identificador de proveedor atípico:",
    awards_df["identificador_proveedor_atipico"].sum()
)

Procedimientos con identificador de comprador atípico: 1
Participaciones con identificador de oferente atípico: 52
Adjudicaciones con identificador de proveedor atípico: 23


In [224]:
# Guardar las bases finales resultantes de la fase de limpieza,
# preprocesamiento y transformación de datos.
#
# Los archivos JSON originales permanecen sin modificaciones en data/raw/2025.
# Estas bases procesadas incorporan los controles de calidad,
# normalizaciones, variables derivadas y criterios metodológicos
# definidos durante la fase 4.2 del proyecto.

ruta_procesados = "../data/processed"

os.makedirs(ruta_procesados, exist_ok=True)

procedures_df.to_csv(
    os.path.join(
        ruta_procesados,
        "01_procedimientos_limpios_2025.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

tender_participation_df.to_csv(
    os.path.join(
        ruta_procesados,
        "02_participacion_oferentes_limpia_2025.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

awards_df.to_csv(
    os.path.join(
        ruta_procesados,
        "03_adjudicaciones_limpias_2025.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print("Bases procesadas guardadas correctamente.")

Bases procesadas guardadas correctamente.
